# Realizing Error Suppression in Partially Fault-Tolerant Quantum Simulations with IBM Quantum Computers - *QDC 2026 Challenge*
This challenge notebook is based on [*Realizing Error Suppression in Partially Fault-Tolerant Quantum Simulations with IBM Quantum Computers* - arXiv:2607.24947](https://arxiv.org/abs/2607.24947). In this challenge, we will explore how quantum error **detection** offers a path for improved performance, even in the current pre-fault-tolerant regime. We will rely on a *partially fault-tolerant* quantum simulation that pairs **fault-tolerant syndrome extraction** with **non-fault-tolerant logical operations**. Focusing on a 2+1D **false-vacuum decay** simulation of the **Mixed-Field Ising model** (MFIM), we will encode **logical qubits** using the $[[4,2,2]]$ **Iceberg code** and employ **Observable-Ranked Postselection** (ORP) to estimate local observables with higher accuracy while avoiding prohibitive shot loss.

*Approx. QPU time: 6 minutes on ibm_boston and 4 s on ibm_phoenix*

### To start, set up a new virtual environment and install the required packages.

This notebook needs **Python 3.14** and Jupyter. You also need an IBM Quantum account (PARTS 3-5 submit jobs to real hardware); PARTS 0-2 run entirely offline.

1. Create and activate an environment:

        # macOS / Linux
        python3.14 -m venv <env-name>
        source <env-name>/bin/activate

        # Windows (PowerShell)
        py -3.14 -m venv <env-name>
        .\<env-name>\Scripts\Activate.ps1

2. Install everything from the pinned requirements file by running the cell below.

   The versions in `requirements.txt` are pinned on purpose. In particular PARTS 3-5 import `qiskit_ibm_runtime.executor_sampler.Sampler`, which **does not exist before qiskit-ibm-runtime 0.50.0** - installing an older runtime leaves you unable to submit any job, 35 points in. The preflight cell that follows checks this for you.


In [ ]:
! pip install -r ../../../requirements.txt
! pip install ipykernel


In [ ]:
# The notebook lives in challenge/; utils/ and data/ sit one level up. Put the root and the
# grader on sys.path and chdir there, so relative paths work in the repo and the kit alike.
import os, sys
from pathlib import Path

_nb = Path.cwd()
_root = next((p for p in (_nb, _nb.parent)
              if (p / "utils").is_dir() and (p / "data").is_dir()), _nb)
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "challenge"))
os.chdir(_root)
print(f"working directory: {_root}")


In [ ]:
"""Preflight: fail here, loudly, rather than two hours in.

Checks the four things that otherwise surface as confusing errors deep in the notebook:
the runtime version (PARTS 3-5), ortools (PARTS 1.4 / 3.2), the grader, and the MPS baseline.
"""
import sys
from importlib.metadata import version, PackageNotFoundError

problems, notes = [], []

if sys.version_info[:2] != (3, 14):
    notes.append(f"this notebook was built and tested on Python 3.14; you are on "
                 f"{sys.version_info.major}.{sys.version_info.minor}")

# PARTS 3-5 need executor_sampler.Sampler, which arrived in qiskit-ibm-runtime 0.50.0.
try:
    rt = version("qiskit-ibm-runtime")
    if tuple(int(p) for p in rt.split(".")[:2]) < (0, 50):
        problems.append(f"qiskit-ibm-runtime {rt} is too old: PARTS 3-5 need >= 0.50.0 for "
                        f"executor_sampler.Sampler. Run  pip install -r requirements.txt")
except PackageNotFoundError:
    problems.append("qiskit-ibm-runtime is not installed")

try:
    from qiskit_ibm_runtime.executor_sampler import Sampler  # noqa: F401
except ImportError as err:
    problems.append(f"cannot import executor_sampler.Sampler ({err}); PARTS 3-5 need it")

try:
    import ortools  # noqa: F401
except ImportError:
    problems.append("ortools is missing: the CP-SAT scheduler in PARTS 1.4 and 3.2 needs it")

try:
    from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex1_2  # noqa: F401
except ImportError as err:
    problems.append(f"cannot import the grader ({err}); run  pip install qc-grader")

import os
if not os.path.exists("data/mps_baseline_L.npz"):
    problems.append("data/mps_baseline_L.npz is missing; PARTS 3-5 are graded against it")

for pkg in ("qiskit", "qiskit-aer", "networkx", "matplotlib"):
    try:
        version(pkg)
    except PackageNotFoundError:
        problems.append(f"{pkg} is not installed")

if problems:
    print("PREFLIGHT FAILED\n")
    for p in problems:
        print(f"  - {p}")
    raise SystemExit(1)

print("Preflight OK.")
print(f"  python             {sys.version.split()[0]}")
print(f"  qiskit             {version('qiskit')}")
print(f"  qiskit-ibm-runtime {version('qiskit-ibm-runtime')}")
print(f"  qiskit-aer         {version('qiskit-aer')}")
for n in notes:
    print(f"\n  note: {n}")
print("\nPARTS 0-2 run offline. PARTS 3-5 additionally need a configured IBM Quantum account.")


## Your team

Set `TEAM_NAME` below and run the cell. Every member of a team uses the same name, and
answers are only graded for a team, so do this before you submit anything.


In [ ]:
try:
    from qc_grader.challenges.qdc_2026 import join_team
except ImportError:
    def join_team(name):
        print(f"qc-grader is not installed, so {name!r} was not registered. "
              "Install it (pip install qc-grader) before submitting.")

TEAM_NAME = ""  # YOUR TEAM NAME, the same for every member

if TEAM_NAME:
    join_team(TEAM_NAME)
else:
    print("Set TEAM_NAME to your team's name and run this cell again: "
          "answers are only graded for a team.")


# **PART 0: Background**
*Theory for reference. Skim it and come back as needed.*


### 0.1 "Partially" Fault-Tolerant
A *partially* fault-tolerant computation mixes fully fault-tolerant (FT) components with
non-fault-tolerant (nFT) ones. Not every physical error is detectable any more, but the circuit
overhead drops a long way -- which is what makes it viable on current hardware. Here the FT
component is syndrome extraction and the nFT components are the logical operations, which would
otherwise need expensive machinery such as magic state injection.

Split the logical error rate into the two contributions, at physical error rate $p$:

- nFT: no suppression, contributes $Ap$
- FT: errors at $\mathcal{O}(p)$ are detected and removed, leaving $Bp^2$

So $p_L(p) = Ap + Bp^2$. Design the nFT part well enough that $A \ll B \ll 1$ and you get
$p_L(p) < p$ -- below the *pseudothreshold*, on today's devices.


### 0.2 The $[[4, 2, 2]]$ Iceberg Code
Iceberg codes are a family of $[[n+2, n, 2]]$ codes that use two extra physical qubits to encode $n$ logical qubits into one code "block." We will focus on the $n=2$ case -- the smallest member of the family -- meaning that we will encode 2 logical qubits per code block. This code is a *Calderbank-Shor-Steane* (CSS) code with "distance" $d=2$, meaning that the code is capable of *detecting* (but not correcting) single-qubit errors. In addition to the four "data" qubits $d_0$-$d_3$ used for logical qubit encoding, each code block includes two "ancilla" qubits $a_x$ and $a_z$ that are used for syndrome readout and logical operations. 

#### *Stabilizers*: 
$$S_X = X_0 X_1 X_2 X_3, \qquad S_Z = Z_0 Z_1 Z_2 Z_3$$
The logical codespace is the space where both $S_X$ and $S_Z$ are $+1$ -- that is, the "joint $+1$-eigenspace" of $S_X$ and $S_Z$. This is a 4-dimensional subspace (enough for 2 logical qubits) of the 16-dimensional four-qubit Hilbert space. The condition $S_Z = +1$ restricts us to the *even-parity sector* (computational basis states with an even number of 1s), and $S_X = +1$ further restricts us to symmetric superpositions within that sector, as seen in the table below. 

#### *Logical basis states*: 
Here, logical basis states are differentiated from physical states by a bar. 

| Logical State | Physical State | Readout Bitstrings ($d_0d_1d_2d_3$) |
| :---: | :---: | :---: |
| $\|\overline{00}\rangle$ | $\frac{1}{\sqrt{2}} \left( \|0000\rangle + \|1111\rangle \right)$ | 0000, 1111 |
| $\|\overline{01}\rangle$ | $\frac{1}{\sqrt{2}} \left( \|0011\rangle + \|1100\rangle \right)$ | 0011, 1100 |
| $\|\overline{10}\rangle$ | $\frac{1}{\sqrt{2}} \left( \|0101\rangle + \|1010\rangle \right)$ | 0101, 1010 |
| $\|\overline{11}\rangle$ | $\frac{1}{\sqrt{2}} \left( \|0110\rangle + \|1001\rangle \right)$ | 0110, 1001 |

Note that $|\overline{00}\rangle$ is a four-qubit GHZ state. Take a moment to look at the different readout bitstrings: every valid bitstring has *even parity*. When we measure the data qubits in the computational basis at the end of a circuit, we will want to postselect readout bitstrings that fall within this even-parity sector. An easy way to check is to see if $d_0 \oplus d_1 \oplus d_2 \oplus d_3 = 0$; we will use this logic later for postselection. Keep in mind that this final-readout check only measures $S_Z$ (i.e. it catches $X$-type errors); $Z$-type errors are invisible in a computational-basis measurement and must instead be caught by measuring $S_X$ with the ancilla qubits during syndrome extraction.

#### *Logical operators*: 
$$\overline{X}_i = X_{i+1} X_3, \quad \overline{Z}_i = Z_0 Z_{i+1}, \quad i = 0, 1.$$

Any operator containing only an *even* number of physical $X\rm s$ and an *even* number of physical $Z\rm s$ (counting $Y = iXZ$ as one of each) commutes with both stabilizers and therefore preserves the codespace. *This means that errors of this form cannot be detected!* Apart from the stabilizers themselves (which act trivially on the codespace), such errors are undetectable logical errors. In particular, any error that the code fails to detect must involve at least two physical qubits.

#### *Detectors*
Syndrome measurements record the eigenvalue of a stabilizer at the time of measurement. From one or more rounds of syndrome measurements, we can construct **detectors** $v_j$, where $v_j = 1$ indicates that an error was detected. (Here, like the authors, we use the index $j$ to label $(g, b, r)$, where $g$ denotes the stabilizer, $b$ denotes the code block, and $r$ denotes the syndrome measurement round). 

For the $[[4, 2, 2]]$ Iceberg Code, there are two stabilizers for each code block, leading to two detectors for each block for each round of syndrome measurements. In addition to this, there is a *boundary detector* for each block that is set by the logical state in which the block is prepared and the basis in which it is measured out. In our case, since we will only measure in the $Z$-basis, we only have one such boundary detector for stabilizer $S_Z$. 

The formation of the detectors depends on whether the ancilla qubits are **reset** between syndrome readout rounds. In the table below, we use the shorthand $j \pm 1 = (g, b, r \pm 1)$. For the first detector $j=1$, syndrome measurements are padded with a reference value of 0; this works because the initial GHZ state fixes both stabilizer eigenvalues. 

| |Syndrome measures... | Detectors | Boundary Detector |
| :---: | :--- | :---: | :---: |
| Reset | Instantaneous stabilizer eigenvalue | $v_j = s_{j-1} \oplus s_j$ | $v_{R+1} = s_R$ |
| No Reset | Running parity of the stabilizer's eigenvalue history |  $v_{j+1} = s_{j+1} \oplus s_{j-1}$| $v_{R+1} = s_{R-1} \oplus s_{R}$|

We have simplified the boundary detectors in the table above because we will always operate with *parity postselected data*, which eliminates the need to add a check on the data qubits themselves. 

Two important comments about the $R = 1$ case, which is where our hardware experiments will focus: 
1. When there is only one readout round ($R = 1$), due to the padding, this reduces the detectors to be $v_1 = s_1$ for all $g$, $b$, 
2. Consequently, the boundary detector reduces to $v_2 = s_1 = v_1$ and becomes redundant. 

### 0.3 Observable-Ranked Postselection (ORP)
Throwing away every shot with a detected error is wasteful. *Observable-ranked postselection*
instead uses the fact that a logical observable cannot be affected by errors *causally
disconnected* from it, or by errors that *commute* with it.

Rank each detector by how strongly it correlates with a logical error on the observable at hand:

$$\Delta_j = |\langle \overline{O} \rangle_{v_j = 0} - \langle \overline{O} \rangle_{v_j = 1}|$$

where $\langle \overline{O} \rangle_{v_j = b}$ is the observable estimated from the shots with
$v_j = b$. Then keep only the shots in which none of the top-$k$ detectors fired. Larger $k$
means less bias and fewer shots, so more variance; PART 2 builds the rule that picks $k$.

$\Delta_j$ is computed **per observable**, so every observable gets its own ranking.


### 0.4 The Simulation Physics

#### *The Mixed-Field Ising Model (MFIM)*
$$\overline{H} = - J\sum_{\langle ij\rangle} \overline{Z}_i \overline{Z}_j
- \sum_i \left(g_x \overline{X}_i + g_z \overline{Z}_i\right)$$

over nearest-neighbor pairs $\langle ij \rangle$, open boundaries. $g_x$ sets the transverse
field, $g_z$ the longitudinal one. The lattice is *almost* square: a few couplings have to be
dropped to fit the code blocks onto heavy-hex.

#### *False-Vacuum Decay*
For $g_z > 0$ the longitudinal field favors $\overline{Z}_i = +1$, so all-$|\overline{0}\rangle$
is the true vacuum and all-$|\overline{1}\rangle$ the false one. We quench a product-state
bubble of true vacuum into a false-vacuum background:

$$|\psi(t=0)\rangle = \prod_{i\in B} \overline{X}_i|\overline{1}\rangle^{\otimes N}$$

with $B$ the initial true-vacuum region. At large $J$, bubbles are predicted to "melt"
(delocalize) for $|g_z| / |g_x| < 1$ and stay localized for $|g_z| / |g_x| > 1$.

#### *Parameters*
$J = 1$, $g_x = 0.75$, and two values of $g_z$:
- $g_z = 0.5$: **bubble-melting regime**
- $g_z = 1.5$: **localized regime**


### 0.5 Embedding on Heavy-Hex Hardware
The final hardware embedding choice determines the efficiency of the workflow, and an optimized embedding for each circuit element is what keeps the logical error rate below the physical error rate. We will replicate the authors' implementation rather than trying to perform this optimization from scratch.

On a heavy-hex graph, a single code block requires one degree-3 vertex and two degree-2 vertices. There are two different optimal arrangements employed for the different workflow elements: 

<table>
  <tr>
    <td width="400">
      <h3><strong>The compute configuration</strong></h3>
      <p>Optimal for <em>logical operations</em>. This arrangement is used during the logical state preparation and Trotter portions of the workflow. </p>
    </td>
    <td width="200">
      <img src="figures/compute_config.png" alt="Figure Caption" width="200">
    </td>
  </tr>
  <tr>
    <td width="400">
      <h3><strong>The syndrome configuration</strong></h3>
      <p>Optimal for <em>syndrome extraction</em>. This arrangement allows for efficient measurement of $S_X$ and $S_Z$ simultaneously. </p>
    </td>
    <td width="200">
      <img src="figures/syndrome_config.png" alt="Figure Caption" width="200">
    </td>
  </tr>
</table>

The large-scale simulation uses a total of 21 code blocks. With two logical qubits in each block, this is a total of 42 logical qubits. The optimal placement of code blocks used by the authors is tailored for `ibm_boston`, avoiding specific noisy qubits and edges. The authors arrived at this placement through simulated annealing over arrangements of the blocks on a square lattice, where two logical qubits are coupled only when their blocks are neighbors on that lattice and physically connected (directly or through one bridge qubit). With the chosen block placement, a nearly square lattice can be realized; a total of 4 edges are missing from a perfectly square lattice. 

To keep track of the code blocks and logical qubits, we will use two integers for labeling: $$(b, l_b) \text{ with } l_b \in \{ 0, 1\}$$

There are a few more concepts that are helpful to introduce to keep track of the blocks and how they are embedded on the hardware. 

#### *Frame* 
We use the term "frame" $f \in \{ 0, 1\}$ to specify the relative position of data qubits $d_1$ and $d_2$, specifically in the compute configuration. 
- $f = 0$: $d_1$ and $d_2$ as pictured above in compute configuration. This arrangement allows direct implementation of $\overline{Z}_{(b, 0)}$ and $\overline{X}_{(b, 1)}$. 
- $f = 1$: $d_1$ and $d_2$ swapped with each other. This arrangement allows direct implementation of $\overline{Z}_{(b, 1)}$ and $\overline{X}_{(b, 0)}$. 

Inter-block gates require at least one frame change per Trotter step.

#### *Relabeling freedom*
Within each code block, there is some freedom in labeling. These choices change nothing logically, but they have consequences in terms of circuit depth, and we'll need to keep track of the choices. 
- $a_x \leftrightarrow a_z$: We use the term "orientation" $o_b \in \{0, 1\}$ to specify whether $a_x$ and $a_z$ have been exchanged. 
- $(b, 0) \leftrightarrow (b, 1)$: We use the variable $l_b \in \{0, 1\}$ to specify whether the two logical qubits within a block have been relabeled. 
- Coupling connection: In some cases, more than one connection could be used to perform inter-block gates. We use the variable $w_e$ to specify which choice is used. 

#### *Scheduling*
There is a large amount of freedom in how the inter-block interactions are scheduled during a Trotter step, and the scheduling choice has a large effect on the circuit depth. The schedule can be parameterized by specifying $A = \{o, l, w\}$ for all blocks and edges in the logical grid. But $A$ only specifies the gate placements; the gate ordering for a fixed $A$ is a combinatorial optimization problem. We will provide code to perform this gate-level optimization using CP-SAT. It is important that the comparison benchmarks (MPS, statevector, and unencoded simulations) use the same gate order, because a different gate order can change the Trotter error of the simulation. 

### 0.6 The Full Picture
1. Initialize the code blocks into $|\overline{00}\rangle$
2. For $R$ repetitions:
    - logical rotations (one Trotter step of the false-vacuum decay)
    - syndrome extraction (measure the ancillas)

<img src="figures/circuit.png" width="600">


<div class="alert alert-block alert-success">

### **Your Task:**
### **Use the $[[4,2,2]]$ Iceberg code to implement a *partially fault-tolerant* quantum simulation of 2+1D false-vacuum decay in the Mixed-Field Ising model (MFIM).**

### Part 1 (Small-Scale Encoded Simulation) -- 40 pts:
- Toy lattice: two code blocks (15 and 12) $\rightarrow$ 4 logical qubits, a single square of the paper's Fig. 2a grid, with 2 true-vacuum bubble sites
- Encode $|0000\rangle \rightarrow |\overline{00}\rangle \rightarrow |\overline{\psi}(t=0)\rangle$, schedule one Trotter step, extract syndromes, measure, and assemble the full circuit
- Check it against a noisy `qiskit-aer` simulation and an exact unencoded statevector baseline
- $n_{\rm steps} = \{0, 1, \ldots, 5\}$, $\delta t = 0.5$, $J = 1$, $g_x = 0.75$, $g_z = 0.5$

### Part 2 (Observable-Ranked Postselection) -- 20 pts:
- Parity postselection, detector construction from the syndromes, detection-event correlations $\Delta_j$, and bootstrapped uncertainties
- Postselect on the top-$k$ detectors, then select the $k$ that balances bias against variance
- Assemble the full ORP pipeline

### Part 3 (Large-Scale Encoded Experiment) -- 20 pts:
- The paper's Fig. 2a lattice: 21 blocks $\rightarrow$ 42 logical qubits on 136 physical qubits of `ibm_boston`
- $n_{\rm steps} = \{0, 1, \ldots, 8\}$ (up to $t = 4$), $R = 1$ syndrome round, 32,000 shots
- Both field regimes: $g_z = 0.5$ (melting) and $g_z = 1.5$ (localized)
- Compare your ORP estimates against the MPS baseline

### Part 4 (Unencoded Baseline) -- 10 pts:
- The same evolution on 42 **physical** qubits: same Trotter order, same shots, no code
- Show whether the encoded result actually beats the unencoded one

### Part 5 (Improvements) -- 10 pts:
- Add dynamical decoupling and measurement twirling to both arms
- Run the unencoded circuits on a square-lattice (Nighthawk) device

</div>

<div class="alert alert-block alert-warning">

### **Grading Criteria:**
Your score comes from **25 graded exercises worth 100 points in total**. Most of them check an
implementation against a reference. The heavily weighted exercises in Parts 3-5 are scored on how
close your **hardware** results come to the MPS baseline, and on whether error detection actually
improved the local observables.

#### **40 Points: Small-scale encoded simulation** (PART 1)
- 2.5 pts for the toy system setup (*PART 1.2*)
- 10 pts for encoded state preparation (*PART 1.3*, 5 sub-functions $\times$ 2.0 pts)
- 2.5 pts for the Trotter step optimization (*PART 1.4*)
- 10 pts for syndrome extraction (*PART 1.5a*, 4 sub-functions $\times$ 2.5 pts)
- 2.5 pts for the final measurements (*PART 1.5b*)
- 5 pts for building the full circuit (*PART 1.6*)
- 2.5 pts for the encoded / unencoded observables (*PART 1.7*)
- 2.5 pts for the noisy simulation (*PART 1.8*)
- 2.5 pts for the unencoded circuit used in the statevector comparison (*PART 1.9*)

#### **20 Points: Observable-ranked postselection** (PART 2)
- 2.5 pts each for parity postselection (*PART 2.1*), detector construction (*PART 2.2*), detection-event correlations (*PART 2.3*), bootstrapped uncertainties (*PART 2.4*), and top-$k$ postselection (*PART 2.5*) (12.5 pts total)
- 2.5 pts for optimal-$k$ selection (*PARTS 2.6–2.7*, combined in `check_part2p7`: 1.25 pts for $O_{\rm ref}$ and 1.25 pts for $k^\star$)
- 5 pts for the full ORP pipeline (*PART 2.8*)

#### **20 Points: Large-scale encoded experiment** (PART 3)
- 2.5 pts each for the full system setup (*PART 3.1*), the Trotter-step optimization (*PART 3.2*), circuit generation and transpilation (*PART 3.3*), and job retrieval and postselection (*PART 3.4*) (10 pts total)
- 10 pts for the median $|\langle \bar{Z} \rangle_{\rm meas} - \langle \bar{Z} \rangle_{\rm MPS}|$ against the MPS baseline (*PART 3.5 / 3.6*, 5 pts per regime)

#### **10 Points: Unencoded baseline** (PART 4)
- 2.5 pts for unencoded circuit generation and transpilation (*PART 4.1*)
- 7.5 pts for the encoded-versus-unencoded comparison (*PART 4.3*, `check_part4p2`): 5.0 for the melting regime, 2.5 for a sound localized measurement

#### **10 Points: Improvements** (PART 5)
- 5 pts for error mitigation (DD + measurement twirling) on both arms (*PART 5.1*)
- 5 pts for the Nighthawk run of the unencoded baseline (*PART 5.2*)

Parts 3-5 are graded on **self-reported hardware numbers**: the grader checks the values you
submit for internal consistency and against its own reference, it cannot re-run your jobs.

#### **Winning Team**
The winning team with the highest total score will have the opportunity to present their solution
notebook to the other contestants.

#### *Good luck and have fun!*

</div>

# **PART 1: Small-Scale Encoded Simulation**
Build every piece of the workflow and test it on a two-block toy system with a noisy simulation.


### 1.1 Establishing the classes and variables
Most of the bookkeeping lives in five modules under `utils/`. You only call a handful of their
functions directly, and each is introduced where it is first used.

- `iceberg_device.py` -- the `ibm_boston` heavy-hex geometry: 156 qubits, the coupling map, the
  drawing coordinates. Used by the other modules, not by you.
- `iceberg_block.py` -- one $[[4,2,2]]$ block: the six qubit roles (`d0`-`d3`, `ax`, `az`), the
  six physical sites it occupies, which role sits where in the compute and syndrome
  configurations, and the logical-operator supports. `LogicalBlock` turns a role plus a frame
  into a *physical qubit index*.
- `iceberg_layout.py` -- the paper's 21-block placement (`PLACEMENT`), the bridge qubits, the
  Fig. 2a logical grid (`FIG2A_GRID`, `FIG2A_MISSING`), and two classes: `LogicalLattice` (the
  Hamiltonian graph -- nodes are logical qubits, edges are $ZZ$ couplings) and `Placement` (the
  physical placement: which qubits each block owns and how blocks connect).
- `iceberg_schedule.py` -- builds and optimizes the encoded Trotter step: the parameters
  $A = \{o, l, w\}$ (`ParamA`), the gate-level schedule, and the logical Trotter order the
  benchmarks reuse.
- `iceberg_viz.py` -- plotting for blocks, placements, logical graphs and results.

Four labels recur throughout:
- **Lattice site** `(b, l)` -- a logical qubit in the Hamiltonian (the paper's $(b, l_b)$).
  Bubble sites, observables and plots all use these.
- **Code qubit** `(b, l')` -- which of block $b$'s two encoded qubits carries that site. Usually
  $l' = l$, but the relabeling choice $l_b$ in $A$ can swap them: $l' = l \oplus l_b$.
  `A.code_qubit(site)` does the conversion.
- `frames` -- one entry per block, 0 or 1, giving that block's current frame (whether $d_1$ and
  $d_2$ are swapped). Every circuit-building function takes it and keeps it current.
- `block_to_index` -- `{block: position}`, mapping a block label to its position in `frames` and
  in the classical registers.


### 1.2 Build the toy system


Before the 136-qubit run, a toy model: two code blocks, 4 logical qubits, one square of the full
lattice. Two of those logical qubits are the "bubble" sites that start in the true vacuum.

The cell below uses:
- `LogicalLattice.from_rows(rows, omit=[])` -- builds the lattice from a `{row: {column: (b, l)}}`
  dictionary. Couplings are all nearest neighbors on that grid, minus any pair in `omit`. The toy
  lattice is the 2x2 square at rows 5-6, columns 5-6 of Fig. 2a, i.e. blocks 15 and 12.
- `PLACEMENT` -- the paper's placement, `{block: [six physical qubits]}`, read off Fig. 1b.
  `block_qubits` keeps only the blocks this lattice uses.
- `Placement.from_qubits(placement=block_qubits)` -- works out which qubit plays which role,
  records the bridge qubits, and defaults to the `ibm_boston` topology. Circuits are built on the
  full 156-qubit register, so physical indices go straight to the hardware.
- `bubble_sites` -- lattice-site labels `(b, l)`.


In [ ]:
from utils.iceberg_layout import LogicalLattice, Placement, PLACEMENT

# TODO: the toy system is the 2x2 square at rows 5-6, columns 5-6 of the paper's Fig. 2a grid.
# It uses two blocks; `from_rows` takes {row: {column: (block, logical)}}.
logical_blocks = ...
lattice = LogicalLattice.from_rows(..., omit=[])

# TODO: two of the four sites start in the true vacuum ("the bubble"), in (block, logical) labels
bubble_sites = ...

# Everything below is derived from your choices above - no need to change it.
block_qubits = {b: PLACEMENT[b] for b in lattice.blocks()} # placement from the paper, but only the blocks in `logical_blocks`
placement = Placement.from_qubits(placement=block_qubits) # defaults for bridges and topology
block_to_index = {b[0]: i for i, b in enumerate(placement.blocks.items())}
num_logical_blocks = len(placement.blocks)

Two views of the toy system: the physical device, and the encoded space as a logical graph.

- `plot_placement(placement, heron=True)` -- data qubits black, `az` red, `ax` blue, bridges
  green, unused gray, with a halo per block. `heron=True` uses the real heavy-hex geometry.
- `plot_logical_graph(lattice.to_graph(), bubble_sites=...)` -- thick edges are intra-block
  couplings, thin edges inter-block. Bubble sites are dark red.


In [ ]:
from utils.iceberg_viz import plot_placement
plot_placement(placement, heron=True);

In [ ]:
from utils.iceberg_viz import plot_logical_graph
plot_logical_graph(lattice.to_graph(), bubble_sites=bubble_sites);

<div class="alert alert-block alert-info">

### **Check your implementation below (2.5 pts).**



<!-- p1-hints -->
<details>
<summary><b>Stuck on 1.2?</b> (click to expand)</summary>

- **Wrong site or edge count** - the toy system is the 2×2 square at rows 5-6, columns 5-6 of the paper's Fig. 2a, with *no* omitted pairs (`omit=[]`).
- **Wrong bubble** - two of the four sites start in the true vacuum.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex1_2
grade_lab_iceberg_ex1_2(lattice, block_qubits, bubble_sites, placement);

### 1.3 Code Initialization: $|0000\rangle \rightarrow |\overline{00}\rangle \rightarrow |\overline{\psi}(t=0)\rangle$
First, the circuit itself: one classical register for the data readout (`'data'`) and one per
syndrome round (`f'syn{i}'`). `frames` starts as a list of zeros -- every block in the untouched
compute configuration.


In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister

def init_circuit(num_logical_blocks, R, nq):
    """
    Create a circuit
    """
    # TODO: one classical register 'syn{i}' of 2 bits per block for each of the R syndrome
    # rounds, one register 'data' of 4 bits per block, and a quantum register 'q' of nq qubits.
    # `frames` starts as one 0 per block: every block begins in the compute configuration.

    return qc, frames

Next, prepare each block's logical qubits into $|\overline{00}\rangle$, which is a 4-qubit GHZ
state on the data qubits. Fill in `prep_logical_zero` for a single block; the following function
`prep_logical_zero_blocks` calls it for every block in a `placement`.

*Each block in `placement.blocks` is a `LogicalBlock`. `block.physical(role, frame)` returns the
physical qubit currently holding `role`, and in `frame=1` the roles `d1` and `d2` trade places.
Always ask for qubits by role and frame -- after the first Trotter step a block's frame changes
and the same role sits elsewhere.*


In [ ]:
def prep_logical_zero(qc, d0, d1, d2, d3): 
    """Prepare a single logical |\bar{00}> state on data qubits d0-d3"""
    # TODO: the four data qubits must end in a GHZ state

    return qc

def prep_logical_zero_blocks(qc, frames, placement, block_to_index, barriers=True):
    """Set logical qubits in all blocks to logical zero state"""

    # TODO: for every block, look up the four data qubits by role at that block's current frame
    # (block.physical("d0", frame=...)) and call prep_logical_zero on them.
    
    if barriers:
        qc.barrier();

    return qc

Now "flip" the logical qubits that start in the false vacuum, which surrounds the true-vacuum
bubble. That needs a *logical $X$ gate*.


In [ ]:
def logical_x(qc, placement, b, l, frame):

    # TODO: apply Xbar_l on block b. Xbar_0 and Xbar_1 each act on two of the data qubits -
    # look up the supports in utils/iceberg_block.py (X_SUPPORTS), by role at this frame.

    return qc

`create_bubble` uses `logical_x` to build the false vacuum around the bubble. It needs the
schedule parameters `param_A` (see 1.4) because of the relabeling freedom:
`param_A.code_qubit(site)` says which encoded qubit of block $b$ carries that lattice site, and
therefore which $\overline{X}$ to apply.


In [ ]:
def create_bubble(qc, frames, placement, lattice, param_A, bubble_sites, block_to_index, barriers=True):
    """
    Initial state: bubble of true vacuum |0> on `bubble_sites`, false vacuum |1> elsewhere. Adding parameterization from A
    """
    # TODO: every lattice site that is NOT a bubble site gets one logical X. Use
    # param_A.code_qubit(site) to find which (block, logical) carries that site.

    if barriers:
        qc.barrier()
    return qc

<div class="alert alert-block alert-info">

### **Check your implementation below (10 pts total).**

The five state-preparation functions are scored together: 2 pts each.



<!-- p1-hints -->
<details>
<summary><b>Stuck on 1.3?</b> (click to expand)</summary>

- **`prep_logical_zero` scores 0** - the four data qubits must end in the GHZ state, and the function must **return** the circuit it built on (not `None`).
- **Right at frame 0, wrong at frame 1** - you are indexing qubits by position instead of by role. Ask for them with `block.physical(role, frame=…)`; which physical qubit holds `d1` depends on the frame.
- **`logical_x` scores 0** - wrong support: `Xbar_0` and `Xbar_1` each act on *two* of the four data qubits.
- **`create_bubble` inverted** - bubble sites stay in $|\overline{0}\rangle$; every *other* site gets one logical X.
- **Register error** - one `syn{i}` register of 2 bits per block, per syndrome round.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex1_3
grade_lab_iceberg_ex1_3(init_circuit, prep_logical_zero, prep_logical_zero_blocks,
                        logical_x, create_bubble);

### 1.4 Trotterization Schedule
The schedule -- the parameterization $A = \{o, l, w\}$ and the gate order within a step -- sets
the circuit depth, and therefore the final accuracy.

`ScheduleOptimizer` in `utils/iceberg_schedule.py` searches for the $A$ with the shallowest
Trotter step by coordinate descent: change one $o_b$, $l_b$ or $w_e$ at a time, keep the change
if the step gets shallower, and score each candidate by solving the gate ordering exactly.
`optimize()` returns:

- `A` (`ParamA`) -- the chosen orientations `o`, relabelings `l` and connections `w`.
- `step` (`TrotterStep`) -- every gate of one Trotter step, plus `step.place` (the `Placement`
  rebuilt with $A$'s orientations) and `step.trotter_order(n)` (the exact sequence of logical
  rotations that $n$ steps implement).
- `gsched` (`GateSchedule`) -- a time slot for every gate. `gsched.depth` is the two-qubit depth
  of one step, `gsched.n_cx` its CNOT count, and `gsched.status` is `OPTIMAL` when the depth was
  proved minimal.


In [ ]:
# Note: this solves a CP-SAT model and prints nothing while it works.
# On the toy lattice it takes about a second.
from utils.iceberg_schedule import ScheduleOptimizer

# TODO: build the optimizer for the toy lattice and run it to convergence.
# optimize() returns (A, step, gsched) - see the description above.
schedule = ...
A, step, gsched = ...

print(f"\none step: depth {gsched.depth} ({gsched.status}), {gsched.n_cx} CNOTs")

Update `placement` and `block_to_index` to match the orientations of the chosen $A$.


In [ ]:
placement, block_to_index = step.place, step.block_to_index

<div class="alert alert-block alert-info">

### **Check your implementation below (2.5 pts).**

Scored on the two-qubit depth your schedule achieves, so a shallower step is worth more. Full marks at the optimum.



<!-- p1-hints -->
<details>
<summary><b>Stuck on 1.4?</b> (click to expand)</summary>

- **Depth above 12** - let `optimize()` run to convergence; it reaches depth 12 with 22 CNOTs on the toy lattice.
- **Status is not `OPTIMAL`** - the depth is then not proven minimal.
- **Grader cannot read it** - pass the `gsched` returned by `ScheduleOptimizer.optimize()`, not the optimizer or the step.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex1_4
grade_lab_iceberg_ex1_4(gsched);

### 1.5 Syndrome Extraction and Data Measurement
The FT half of the workflow: measure the ancillas to detect errors. That means shuffling qubits
from the "compute" configuration into the "syndrome" configuration and back. Fill in the
functions below with SWAP gates; the circuits from the paper are shown for reference.

<table>
  <tr>
    <td width="250">
      <h3><strong>Compute -> Syndrome</strong></h3>
      <p> </p>
    </td>
    <td width="400">
      <h3><strong>Syndrome Extraction</strong></h3>
      <p> </p>
    </td>
    <td width="250">
      <h3><strong>Syndrome -> Compute</strong></h3>
      <p> </p>
    </td>
  </tr>
  <tr>
    <td width="250">
      <img src="figures/compute_to_syn.png" alt="Compute to syndrome" width="180">
    </td>
    <td width="400">
      <img src="figures/syndrome_extraction.png" alt="Syndrome extraction" width="400">
    </td>
    <td width="250">
      <img src="figures/syn_to_compute.png" alt="Syndrome to compute" width="140">
    </td>
  </tr>
</table>


In [ ]:
def compute_to_syndrome(qc, block, frame): 
    az, ax, d1, d2, d3 = (block.physical(q, frame) for q in ("az", "ax", "d1", "d2", "d3"))
    # TODO: SWAP into the syndrome configuration, following the left-hand figure above.
    
def syndrome_to_compute(qc, block, frame): 
    az, ax, d1, d2, d3 = (block.physical(q, frame) for q in ("az", "ax", "d1", "d2", "d3"))
    # TODO: SWAP back into the compute configuration (right-hand figure above).

Now the measurement itself. Afterwards the ancillas are either reset to $|0\rangle$
(`reset_ancillas = True`) or left in their collapsed state (`reset_ancillas = False`).


In [ ]:
from qiskit.circuit import Measure
def measure_syndromes(qc, i, block, frame, round, reset_ancillas): 
    az, ax, d0, d1, d2, d3 = (block.physical(q, frame)
                                for q in ("az", "ax", "d0", "d1", "d2", "d3"))
    # TODO: build the S_Z / S_X extraction gadget of the middle figure above, then measure the
    # two syndrome bits of this block: bit 2*i + 0 is S_Z and bit 2*i + 1 is S_X, both in the
    # classical register of this `round` (qc.cregs[round]).

    if reset_ancillas:
        # TODO: reset the two measured qubits so they are back in |0> for the next round
        pass

`syndrome_extraction` puts it together: move into the syndrome configuration, measure the
ancillas, move back. The move back is only needed if more logical operations follow -- that is
what the `terminal` flag tracks.


In [ ]:
def syndrome_extraction(qc, place, round, terminal=False, barriers=True, reset_ancillas=False):
    """One round of S_X / S_Z measurement on every block (as in the workflow notebook).

    ``reset_ancillas``: the gadgets route parity through the ``ax`` / ``az`` ancillas, which is
    only correct if they hold |0>. After measurement they hold the syndrome bit, so a syndrome of
    1 would flip the sign of every rotation routed through that ancilla. Resetting the two
    measured qubits restores |0>. The paper does not reset.
    """
    frame = 0   # S_X and S_Z are symmetric under d1 <-> d2, so the frame does not matter

    # TODO: swap every block into the syndrome configuration, measure the syndromes of every
    # block, and then swap back - but only if this round is not `terminal`, because a terminal
    # round is followed by the data measurement, not by another Trotter step.

    return qc

<div class="alert alert-block alert-info">

### **Check your implementation below (10 pts total).**

The four syndrome-extraction functions are scored together: 2.5 pts each.



<!-- p1-hints -->
<details>
<summary><b>Stuck on 1.5a?</b> (click to expand)</summary>

- **SWAP order wrong** - the sequence must match the figures above, gate for gate.
- **Syndrome bits swapped** - bit `2i+0` is $S_Z$ and bit `2i+1` is $S_X$.
- **Terminal round** - a terminal round *stays* in the syndrome configuration; it does not swap back.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex1_5a
grade_lab_iceberg_ex1_5a(compute_to_syndrome, syndrome_to_compute, measure_syndromes,
                         syndrome_extraction);

At the end of the circuit the data qubits are measured. Which physical qubit maps to which
classical bit depends on the block's current *frame* and *configuration*, so three helpers from
`iceberg_block` translate between roles and sites:

- `site_of_role(role, configuration)` -- the site (0-5) holding `role` in the "compute" or
  "syndrome" configuration.
- `other_role(role)` -- the partner role under a frame toggle (`d1` <-> `d2`, everything else
  maps to itself).
- `block.site_to_physical()` -- `{site: physical qubit}` for the block.


In [ ]:
from utils.iceberg_block import site_of_role, other_role
    
def final_measurements(qc, place, frames, configuration="compute"):
    """Measure d0..d3 of every block, by role at its current frame, into the ``data`` register
    (bit 4 * i + k = role d_k of the i-th block)."""
    # TODO: for the i-th block, measure roles d0..d3 into data bits 4*i + k. In the "compute"
    # configuration the role sits on block.physical(role, frames[i]); in the "syndrome"
    # configuration use block.site_to_physical() together with site_of_role / other_role.

    return qc

<div class="alert alert-block alert-info">

### **Check your implementation below (2.5 pts).**



<!-- p1-hints -->
<details>
<summary><b>Stuck on 1.5b?</b> (click to expand)</summary>

- **Wrong role order** - bit `4i+k` holds role `d_k` of the *i*-th block at that block's current frame.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex1_5b
grade_lab_iceberg_ex1_5b(final_measurements);

### 1.6 Full Circuit
`full_circuit` takes a `circ_elements` list of the strings `'trotter'` and `'syndrome'`, so you
can vary how many Trotter steps and syndrome extractions the circuit has and where they fall.

Each Trotter step reuses the optimized gate order in `gsched`, reversed every other step (as in
the paper) to reduce Trotter error.


In [ ]:
import numpy as np 

def full_circuit(gsched, A, bubble_sites, circ_elements, lattice, placement, block_to_index, J=1.0, gz=0.5, gx=0.75, delta_t=0.5, reset_ancillas=False, barriers=False):
    
    num_logical_blocks = len(placement.blocks)
    R = (np.array(circ_elements) == 'syndrome').sum()
    num_trotter = (np.array(circ_elements) == 'trotter').sum()
    print(f"{R} syndrome rounds, {num_trotter} Trotter steps")

    # TODO: assemble the circuit from the pieces you just built, in order:
    #   1. init_circuit  2. prep_logical_zero_blocks  3. create_bubble
    #   4. walk `circ_elements` in order: 'trotter' -> gsched.append_step(...), with
    #      reverse=True on every second step; 'syndrome' -> syndrome_extraction(...), with
    #      terminal=True only for the last element
    #   5. final_measurements - in the "syndrome" configuration if the last element was a
    #      syndrome round, otherwise in "compute"
    # Remember that append_step returns the updated `frames`.

    return qc

Set the Hamiltonian parameters and `delta_t = 0.5`, as in the paper.

Try `barriers = True` to see the pieces of the circuit: state preparation, bubble creation, a
Trotter step, the move into the syndrome configuration, syndrome extraction, final measurement.


In [ ]:
J = 1
gz_melting = 0.5
gz_localized = 0.5
gx = 0.75
delta_t = 0.5

barriers = True
reset_ancillas = False

circ_elements = ['trotter', 'syndrome']
qc = full_circuit(gsched, A, bubble_sites, circ_elements, lattice, placement, block_to_index, barriers=barriers)

qc.draw("mpl", idle_wires=False)

<div class="alert alert-block alert-info">

### **Check your implementation below (5 pts).**

Your `full_circuit` is driven with a stand-in schedule, so this scores the assembly order and the `reverse` flag, not the gate ordering.



<!-- p1-hints -->
<details>
<summary><b>Stuck on 1.6?</b> (click to expand)</summary>

- **A step is missing** - the order is: init → prep $|\overline{00}\rangle$ → bubble → the elements in order → final measurements.
- **`reverse` never set** - every second step runs with `reverse=True`.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex1_6
grade_lab_iceberg_ex1_6(full_circuit);

### 1.7 Computing Observables
The observable of interest is the logical $Z$ expectation value $\langle \overline{Z} \rangle$ on
each logical qubit.

Each block offers two ways to measure it. `Z_SUPPORTS[l]` lists the two data-qubit pairs
representing $\overline{Z}_l$: `(d0, d1)` or `(d2, d3)` for $l=0$, and `(d0, d2)` or `(d1, d3)`
for $l=1$. The two differ by $S_Z$, so on parity-even shots they agree. `build_observables` makes
one observable per representative (`rep = 0, 1`). Data bit `4i + k` holds role `d_k` of block `i`,
because `final_measurements` measures by role at the final frame.

Build two dictionaries of single-site $Z$ observables: one *encoded*, one *unencoded* (plain
physical qubits, no code).


In [ ]:
from qiskit.quantum_info import SparsePauliOp
from utils.iceberg_block import Z_SUPPORTS   # Z_SUPPORTS[l] = both representatives of Zbar_l, by role

def build_observables(placement, lattice, A):
    """<Zbar> observables for the encoded and the unencoded circuit.

    Returns
    -------
    encoded : dict
        ``(site, rep) -> SparsePauliOp`` on the encoded circuit's `data` register, one entry per
        stabilizer-equivalent representative (rep = 0, 1) of each site's Zbar.
    unencoded : dict
        ``site -> SparsePauliOp`` (Z on that site) on the unencoded circuit, whose qubit k carries
        ``sites[k]`` (default: lattice.nodes(); must match the unencoded circuit).
    """
    sites = list(lattice.nodes())
    n_bits = 4 * len(placement.blocks)
    index = {b: i for i, b in enumerate(placement.blocks)}
    encoded_obs, unencoded_obs = {}, {}
    # TODO: for each site, A.code_qubit(site) gives its (block, logical). Build one ZZ observable
    # per representative in Z_SUPPORTS[l] on data bits 4*index[b] + k (k from the role name), and
    # one single-Z unencoded observable on that site's qubit.

    return encoded_obs, unencoded_obs

Let's go ahead and build these observables for the toy system. 

In [ ]:
encoded_obs, unencoded_obs = build_observables(placement, lattice, A)

<div class="alert alert-block alert-info">

### **Check your implementation below (2.5 pts).**



<!-- p1-hints -->
<details>
<summary><b>Stuck on 1.7?</b> (click to expand)</summary>

- **Single `Z` instead of `ZZ`** - $\bar{Z}_l$ is the ZZ pair from `Z_SUPPORTS[l]`, on data bits `4i+k`.
- **Too few entries** - one entry per (site, representative), not one per site.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex1_7
grade_lab_iceberg_ex1_7(build_observables);

### 1.8 Noisy Simulation
Build circuits for `[0, 1, 2, 3, 4, 5]` Trotter steps, with one syndrome extraction whenever
there is at least one Trotter step.


In [ ]:
qcs = []
n_steps_list = [0, 1, 2, 3, 4, 5]

for n_steps in n_steps_list: 
    if n_steps > 0: 
        circ_elements = ['trotter'] * n_steps + ['syndrome']
    else: 
        circ_elements = []
    qc = full_circuit(gsched, A, bubble_sites, circ_elements, lattice, placement, block_to_index, barriers=False)
    qcs.append(qc)
    

Simulate a noisy run. The depolarizing error on `cx` is deliberately large, to make the effect
of error detection visible. All `"swap"` gates are decomposed into `cx` first so they pick up the
same error.


In [ ]:

from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_aer.noise import NoiseModel, depolarizing_error

n_shots = 4000

# TODO: a depolarizing error of 0.05 on every two-qubit 'cx'
noise_model = NoiseModel()

# TODO: decompose the SWAPs first - otherwise Aer never applies the 'cx' noise to them
qcs_noisy = ...

# TODO: run the sampler with the noise model attached
sampler = ...
result_noisy = ...

Separate the classical registers. `data_bitstrings_list` holds the data-qubit bitstrings;
`syn_bitstrings_list` is a list of lists, since a circuit can have more than one readout round.


In [ ]:
data_bitstrings_list = []
data_bitarray_list = []

syn_bitstrings_list = []

for result in result_noisy:
    syn_bitstrings = []
    for reg_name, bitarray in result.data.items(): 
        if reg_name == 'data': 
            data_bitstrings = bitarray.get_bitstrings()
            data_bitarray = bitarray
        else: 
            syn_bitstrings.append(bitarray.get_bitstrings())
    
    data_bitstrings_list.append(data_bitstrings)
    data_bitarray_list.append(data_bitarray)
    
    syn_bitstrings_list.append(syn_bitstrings)

Run the cell below to see how the single-site $Z$ observables change as the bubble "melts." This plot includes both ways to measure $\langle \overline{Z} \rangle$ for each logical qubit.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

colors = ['blue', 'red', 'green', 'purple']
color_of = dict(zip(lattice.nodes(), colors))   # one color per lattice site
marker_of = {0: 'd', 1: 'o'}                     # one marker per representative

fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
for (site, rep), obs in encoded_obs.items():
    exp_vals = [data_bitarray.expectation_values(obs) for data_bitarray in data_bitarray_list]
    ax.scatter(n_steps_list, exp_vals, color=color_of[site], marker=marker_of[rep],
               label=f"{site}, rep {rep}")

site_handles = [Patch(facecolor=color_of[s], label=str(s)) for s in lattice.nodes()]
method_handles = [Line2D([], [], color='black', marker=marker_of[0], linestyle='none', label='rep 0'),
                  Line2D([], [], color='black', marker=marker_of[1], linestyle='none', label='rep 1')]
handles = site_handles + method_handles
ax.legend(handles=handles, ncol=2, loc='upper right')
ax.grid()
ax.set_ylabel(r'$\langle \bar Z \rangle$')
ax.set_xlabel('n_steps')

Before scoring, collect the logical-$Z$ expectation values per Trotter depth.

In [ ]:
# Collect <Zbar> for the grader: one dict per Trotter depth, keyed by (site, rep).
part1_noisy_obs = [
    {key: float(np.real(data_bitarray.expectation_values(obs)))
     for key, obs in encoded_obs.items()}
    for data_bitarray in data_bitarray_list
]

<div class="alert alert-block alert-info">

### **Check your implementation below (2.5 pts).**

The Aer sampler is not seeded, so this is scored on the shape of the decay: it must start at 1 and damp monotonically as the circuit deepens.



<!-- p1-hints -->
<details>
<summary><b>Stuck on 1.8?</b> (click to expand)</summary>

- **Flat curve** - depolarizing 0.05 on every CNOT should damp the signal by at least 0.25. Is the noise model actually attached, and did you decompose the SWAPs? Aer applies `cx` noise only to real `cx` gates.
- **Starts below 0.9** - check the bubble preparation, and that bit `4i+k` really holds role `d_k`.
- **Values above 1** - an expectation value of a Pauli observable cannot exceed 1.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex1_8
grade_lab_iceberg_ex1_8(part1_noisy_obs);

### 1.9 Statevector Baseline
A noiseless reference, so the only error is Trotter error. Same step size and same gate order,
for a fair comparison.

*`step.trotter_order(n_steps)` returns the logical product formula the encoded circuit implements,
as terms like `("x", site)`, `("z", site)` and `("zz", u, v)`. Building the unencoded reference
from that same list gives it the same Trotter error, so any difference between the two comes from
noise rather than term ordering. (`GateSchedule.verify` checks the encoded circuit matches.)*


In [ ]:
def unencoded_trotter_circuit(order, lattice, bubble_sites, delta_t, J, gx, gz, measure=True):
    """Plain (unencoded) circuit for a Trotter order: one qubit per lattice site.

    Qubit k carries ``sites[k]``, so ``sites`` sets the qubit ordering (e.g. `snake_order` for an
    MPS). Sites start in |0> if in ``bubble_sites`` and |1> otherwise. Terms become
    ``rx(-delta_t gx)``, ``rz(-delta_t gz)`` and ``rzz(-delta_t J)``, i.e. exp(-i delta_t/2 H_term)
    (the paper's convention, matching `GateSchedule.append_step`).

    Parameters
    ----------
    order : list of list of tuple
        From `trotter_order`.
    measure : bool
        Append a final measurement of every qubit.

    Returns
    -------
    QuantumCircuit
    """
    sites = list(lattice.nodes())
    idx = {s: k for k, s in enumerate(sites)}
    qc = QuantumCircuit(len(sites))
    # TODO: flip every site that is not in the bubble to |1>, then walk `order` and apply
    # rx / rz / rzz for the "x", "z" and two-site terms. Measure every qubit if `measure`.

    return qc

Run the cell below to add the statevector reference to the plot.

In [ ]:
from qiskit.quantum_info import Statevector

states = []
for n_steps in n_steps_list: 
    qc = unencoded_trotter_circuit(step.trotter_order(n_steps), lattice, bubble_sites, delta_t, J, gx, gz_melting, measure=False)
    states.append(Statevector(qc))


ref_marker = '*'
fig, ax = plt.subplots(figsize=(6, 4), dpi=300, layout="constrained")
for (site, rep), obs in encoded_obs.items():
    exp_vals = [data_bitarray.expectation_values(obs) for data_bitarray in data_bitarray_list]
    ax.scatter(n_steps_list, exp_vals, color=color_of[site], marker=marker_of[rep])

for site, obs in unencoded_obs.items():
    exp_vals = [state.expectation_value(obs).real for state in states]
    ax.scatter(n_steps_list, exp_vals, color=color_of[site], marker=ref_marker)

site_handles = [Patch(facecolor=color_of[s], label=str(s)) for s in lattice.nodes()]
method_handles = [Line2D([], [], color='black', marker=marker_of[0], linestyle='none', label='rep 0'),
                  Line2D([], [], color='black', marker=marker_of[1], linestyle='none', label='rep 1'),
                  Line2D([], [], color='black', marker=ref_marker, linestyle='none', label='ref')]      
handles = site_handles + method_handles
ax.legend(handles=handles, ncol=2, loc='upper right')
ax.grid()
ax.set_ylabel(r'$\langle \bar Z \rangle$')
ax.set_xlabel('n_steps');


<div class="alert alert-block alert-info">

### **Check your implementation below (2.5 pts).**



<!-- p1-hints -->
<details>
<summary><b>Stuck on 1.9?</b> (click to expand)</summary>

- **No false-vacuum flips** - sites *outside* the bubble start in $|1\rangle$.
- **Wrong rotation sign** - compare against the same Trotter order the encoded circuits use.
- **`measure` ignored** - when asked, every qubit must be measured.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex1_9
grade_lab_iceberg_ex1_9(unencoded_trotter_circuit);

# **PART 2: Observable-Ranked Postselection**
Use the measured syndromes to implement ORP.


### 2.1 Parity Postselection

The simpler of the two stabilizers is a parity check: across a block's four data qubits,
$S_Z = Z_{d_0} Z_{d_1} Z_{d_2} Z_{d_3}$ must read $+1$. A shot whose data bits have **odd** parity
in **any** block carries a detected error, so discard it before anything else.

This is the bluntest filter available and it is free -- it needs only the data register. It also
scales worst: one faulty qubit anywhere kills the whole shot, so across 21 blocks the surviving
fraction falls fast. That is why the rest of PART 2 exists.


In [ ]:
from qiskit.primitives import BitArray

def postselect_parity(data_bitstrings_list, syn_bitstrings_list):
    """Drop every shot in which any block reads odd parity over its four data bits."""
    data_bitstrings_ps_list = []
    syn_bitstrings_ps_list = []

    # TODO: a shot survives only if EVERY block has even parity over its four data bits
    # (bits 4*b .. 4*b+3). `BitArray.from_samples(...)` then `.slice_bits([...]).bitcount()`
    # gives you the number of 1s per shot without any string indexing.
    # Discard the violating shots from the data AND from every round of syndrome bitstrings,
    # so that shot i of one still lines up with shot i of the other.

    return data_bitstrings_ps_list, syn_bitstrings_ps_list


In [ ]:
data_bitstrings_ps_list, syn_bitstrings_ps_list = postselect_parity(data_bitstrings_list, syn_bitstrings_list)

<div class="alert alert-block alert-info">

### **Check your implementation below (2.5 pts).**



<!-- p1-hints -->
<details>
<summary><b>Stuck on 2.1?</b> (click to expand)</summary>

- **Too many shots survive** - a shot is discarded when **any** block has odd parity over its four data bits.
- **Syndromes not filtered** - they must be filtered with the *same* mask as the data, round by round.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex2_1
grade_lab_iceberg_ex2_1(postselect_parity);

### 2.2 Detector Construction from Syndromes

A **detector** is a syndrome bit whose value is predictable when there are no errors; a
**detection event** is that bit firing anyway. At $R = 1$ each block contributes two detectors,
$S_X$ and $S_Z$, so the 21-block system has 42.

Build one boolean mask per detector over the shots that survived 2.1: entry $i$ is `True` when
that detector fired in shot $i$. Everything downstream is set algebra on these masks, so shot $i$
of the syndromes must still line up with shot $i$ of the data -- which is what 2.1 preserved by
filtering both with the same mask.


In [ ]:
def detector_violations(syn_bitstrings_ps, num_logical_blocks, reset_ancillas=False):
    """One boolean mask per detector: True on the shots where that syndrome bit fired."""
    if reset_ancillas:
        print("not implemented")
        return

    violation_masks = {}
    for r, syn_bitstrings_ps_r in enumerate(syn_bitstrings_ps):
        if len(syn_bitstrings_ps) != 1:      # R = 1 only; see 0.2 for the R > 1 detectors
            print("not implemented")
            continue
        # TODO: one detector per syndrome bit (2 per block). For detector v, build a boolean
        # mask over shots that is True where bit v fired, and record
        #     violation_masks[v] = {"r": r, "mask": ..., "n_ps": ...}
        # where n_ps counts the shots that would SURVIVE a cut on this detector.

    return violation_masks


<div class="alert alert-block alert-info">

### **Check your implementation below (2.5 pts).**



<!-- p1-hints -->
<details>
<summary><b>Stuck on 2.2?</b> (click to expand)</summary>

- **Mask inverted** - entry `i` is `True` when bit `v` of shot `i` is `'1'`.
- **`n_ps` wrong** - it counts the shots that would *survive*, not the number of detector firings.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex2_2
grade_lab_iceberg_ex2_2(detector_violations);

### 2.3 Compute Correlation Between Detection Events

Cutting on all 42 detectors would leave almost no shots, and most of that sacrifice buys nothing:
a detector on the far side of the lattice is **causally disconnected** from the observable being
estimated, so its firing says nothing about whether *this* $\langle \bar{Z} \rangle$ is biased.

So measure the relevance directly, per observable:

$$\Delta_j = \left| \langle \bar{O} \rangle_{v_j = 0} - \langle \bar{O} \rangle_{v_j = 1} \right|$$

Split the shots by whether detector $j$ fired, estimate the observable on each half, take the gap.
A detector that matters moves the observable and scores a large $\Delta_j$; an irrelevant one sits
near zero and ranks last. Each of the 84 observables gets its own ranking of the same 42
detectors.


In [ ]:
def compute_correlations(encoded_obs, data_bitstrings_ps, violation_masks, num_logical_blocks):
    """Delta_j = |<O>_{v=0} - <O>_{v=1}| for every observable and every detector."""
    correlations = {key: np.zeros(len(violation_masks)) for key in encoded_obs}

    # TODO: for every detector v, split the shots into those where it fired and those where it
    # did not, and compute
    #     Delta_j = |<O>_{v=0} - <O>_{v=1}|
    # for every observable. `BitArray.slice_shots(mask)` takes a boolean mask, and
    # `.expectation_values(obs)` reads the observable off the surviving shots. Leave the entry
    # at 0 when one of the two subsets is empty.

    return correlations


In [ ]:
data_bitstrings_ps = data_bitstrings_ps_list[1]
syn_bitstrings_ps = syn_bitstrings_ps_list[1]

violation_masks = detector_violations(syn_bitstrings_ps, num_logical_blocks)
correlations = compute_correlations(encoded_obs, data_bitstrings_ps, violation_masks, num_logical_blocks)

<div class="alert alert-block alert-info">

### **Check your implementation below (2.5 pts).**

A detector that fires independently of the data has $\Delta_j \approx 0$ and must rank last - that ranking is what the next section cuts on.



<!-- p1-hints -->
<details>
<summary><b>Stuck on 2.3?</b> (click to expand)</summary>

- **Sign error** - $\Delta_j$ is an **absolute** value.
- **Wrong split** - for each detector, split the shots into those where it fired and those where it did not.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex2_3
grade_lab_iceberg_ex2_3(compute_correlations);

### 2.4 Compute Bootstrapped Uncertainties

Postselection trades bias for variance: every discarded shot makes the estimate less biased and
noisier. Finding where that trade stops paying needs the noise, not just the value.

For a logical $\bar{Z}$ every shot returns $\pm 1$, so the estimator is a rescaled binomial
proportion and the bootstrap has a closed form: with $p = (1 + \langle \bar{O} \rangle)/2$ the
fraction of $+1$ shots, resampling $N$ shots with replacement means drawing
$\mathrm{Binomial}(N, p)$ and mapping back through $2\,\text{count}/N - 1$.

This variance is a *statistical* error bar only -- it says nothing about the bias left by
undetected errors, which is why 2.6 and 2.7 need a reference value as well.


In [ ]:
def bootstrap_obs_uncertainty(obs, bit_array_ps, n_samples=1000, rng=None):
    """Bootstrapped variance of <obs> over the shots in `bit_array_ps`.

    `obs` is a single Pauli (Z...Z) term, so every shot gives +1 or -1. Resampling the N shots
    with replacement then gives Binomial(N, p) shots equal to +1, with p the fraction of +1 shots,
    so drawing that count directly is the same bootstrap without building resampled BitArrays.
    """
    rng = np.random.default_rng() if rng is None else rng
    num_shots = bit_array_ps.num_shots
    # TODO: turn <obs> into the fraction p of +1 shots, draw n_samples resampled counts with
    # rng.binomial, map each back to an expectation value, and return their variance.

    return sig2

<div class="alert alert-block alert-info">

### **Check your implementation below (2.5 pts).**

Checked at a fixed seed, and for the right trend: discarding shots must *increase* the variance. That is the cost side of postselection.



<!-- p1-hints -->
<details>
<summary><b>Stuck on 2.4?</b> (click to expand)</summary>

- **Off by a square root** - the bootstrap is Binomial($N$, $p$) with $p = (1 + \langle O \rangle)/2$, and you return the **variance**, not the standard deviation.
- **Constant variance** - it should *grow* when shots are discarded. That trade-off is exactly what $k$ balances.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex2_4
grade_lab_iceberg_ex2_4(bootstrap_obs_uncertainty);

### 2.5 Postselected Observables for all $k$

Compute the whole trade-off curve instead of guessing where to stop. Rank the detectors by
decreasing $\Delta_j$ and, for every cut level $k = 0, 1, \ldots, 42$, keep only the shots in
which **none of the top-$k$ detectors fired**. Record three things per level:

| | |
|---|---|
| $f_k$ | fraction of parity-even shots still surviving -- the **signal survival fraction** |
| $O_k$ | the observable estimated on those survivors |
| $\sigma^2_k$ | its bootstrapped variance from 2.4 |

$f_0 = 1$ by construction. As $k$ grows, $O_k$ should drift toward the true value and then stop
moving, while $\sigma^2_k$ grows without bound as the ensemble empties. 2.6 and 2.7 read those
two curves.


In [ ]:
from functools import reduce

def compute_obs_k(encoded_obs, data_bitstrings_ps, violation_masks, correlations, num_logical_blocks, n_samples=1000):
    num_parity_even_shots = len(data_bitstrings_ps)
    num_det = len(violation_masks.items())
    obs_k = {key: {"O_k": np.zeros(num_det + 1), "f_k": np.zeros(num_det + 1), "sig2_k": np.zeros(num_det + 1)} for key in encoded_obs}
    # TODO: for each observable, rank the detectors by DECREASING Delta, then for every
    # k = 0 ... num_det keep only the shots where none of the top-k detectors fired, and record
    #   f_k[k]     the surviving fraction of the parity-even shots (so f_k[0] = 1)
    #   O_k[k]     <obs> over the survivors
    #   sig2_k[k]  its bootstrapped variance (bootstrap_obs_uncertainty)
    # If a cut leaves no shots, store NaN for O_k and sig2_k at that k.
        
    return obs_k

In [ ]:
obs_k = compute_obs_k(encoded_obs, data_bitstrings_ps, violation_masks, correlations, num_logical_blocks, n_samples=1000)

<div class="alert alert-block alert-info">

### **Check your implementation below (2.5 pts).**



<!-- p1-hints -->
<details>
<summary><b>Stuck on 2.5?</b> (click to expand)</summary>

- **Ranking backwards** - detectors are cut in order of **decreasing** $\Delta$.
- **`f_k` not normalized** - it is the share of the *parity-even* shots left, so `f_k[0] = 1`.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex2_5
grade_lab_iceberg_ex2_5(compute_obs_k);

### 2.6 Build Reference Values for Observables

Choosing $k$ is circular: knowing which levels are unbiased needs the unbiased answer, which is
what we are measuring.

ORP's way out is to trust the **deep cuts**. At small $f_k$ almost every detected error is gone,
so those levels are nearly unbiased -- just noisy. An inverse-variance weighted average over
several of them gives a reference $O_{\rm ref}$ good enough to judge the shallower levels:

$$O_{\rm ref} = \frac{\sum_k O_k / \sigma^2_k}{\sum_k 1/\sigma^2_k}, \qquad
f_{\min} \le f_k \le f_{\rm ref}$$

The window $[f_{\min}, f_{\rm ref}]$ selects "deep but not empty", and its values are
system-dependent: $[0.05, 0.5]$ on the toy system, $[0.005, 0.05]$ in PART 3 where 42 detectors
bite much harder.


In [ ]:
def compute_obs_ref(obs_k, f_min, f_ref, R=1): 
    for key, obs in obs_k.items():
        f_k = obs["f_k"]
        O_k = obs["O_k"]
        sig2_k = obs["sig2_k"]
        
        # TODO: the reference value is the inverse-variance weighted mean of O_k over the levels
        # whose surviving fraction f_k lies inside the window [f_min, f_ref] (and that have a
        # finite O_k and a positive variance). Store it as obs["O_ref"], or NaN when R == 0 or
        # the window is empty.
        
    return obs_k

In [ ]:
for key, obs in obs_k.items():
    f_k = obs["f_k"]
    print(f_k)

In [ ]:
# These window values are system-specific: they are retuned for the
# 21-block run in PART 3.4, where 42 detectors make the window much tighter.
f_min = 0.05
f_ref = 0.5
obs_k = compute_obs_ref(obs_k, f_min, f_ref)

In [ ]:
for key, obs in obs_k.items():
    O_ref = obs["O_ref"]
    print(O_ref)

### 2.7 Select optimal $k$

With a reference in hand: level $k$ is **on the plateau** when it agrees with the reference inside
its own error bar,

$$\left| O_k - O_{\rm ref} \right| \le z \sqrt{\sigma^2_k},$$

and $k^\star$ is the **start of the longest run** of consecutive plateau levels. Below the plateau
bias dominates (the estimate is still moving), above it variance does (the error bars have
swallowed the signal). Requiring a *run* of `target_len` levels rather than a single agreement
stops one lucky level being mistaken for convergence.

If no run is long enough, fall back to the single $k$ minimizing
$(O_k - O_{\rm ref})^2 + \sigma^2_k$.


In [ ]:
def select_k(obs_k, z=1, target_len=2): 
    for key, obs in obs_k.items():
        O_ref = obs["O_ref"]
        O_k = obs["O_k"]
        sig2_k = obs["sig2_k"]
        if not np.isfinite(O_ref):          # no syndrome rounds / empty reference window: parity only
            obs["k_star"] = 0
            obs["O_k_star"] = O_k[0]
            continue
        # TODO: a level k is "on the plateau" when |O_k - O_ref| <= z * sqrt(sig2_k[k]).
        # Walk k downwards, find the longest run of consecutive plateau levels, and take k* at
        # its start. Skip levels whose O_k or sig2_k is NaN. If no run reaches `target_len`,
        # fall back to the k that minimizes (O_k - O_ref)**2 + sig2_k.
                    
        obs["k_star"] = k_star
        obs["O_k_star"] = O_k[k_star]
    
    return obs_k

In [ ]:
obs_k = select_k(obs_k, z=1, target_len=2)

In [ ]:
for key, obs in obs_k.items():
    k_star = obs["k_star"]
    print(k_star)

<div class="alert alert-block alert-info">

### **Check your implementation below (2.5 pts).**

2.6 and 2.7 are scored together: 1.25 pts for $O_{\rm ref}$ and 1.25 pts for $k^\star$.



<!-- p1-hints -->
<details>
<summary><b>Stuck on 2.6 / 2.7?</b> (click to expand)</summary>

- **Unweighted mean** - $O_{\rm ref}$ is the **inverse-variance weighted** mean of $O_k$ over the levels whose $f_k$ lies inside $[f_{\min}, f_{\rm ref}]$.
- **Plateau search skipped** - take the start of the longest run of levels on the plateau; fall back to minimizing $(O_k - O_{\rm ref})^2 + \sigma^2_k$.
- Note the window values are system-specific - they are deliberately different for the 21-block run in PART 3.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex2_6
grade_lab_iceberg_ex2_6(compute_obs_ref, select_k);

### 2.8 Full ORP Pipeline

Everything from 2.1 to 2.7 in one function. PARTS 3, 4 and 5 each run it again -- twice for the
two field regimes, and once more for the improved re-run.


In [ ]:
def run_orp(encoded_obs, data_bitstrings_list, syn_bitstrings_list, num_logical_blocks,
            n_samples=1000, f_min=0.05, f_ref=0.5, z=1, target_len=2):
    """The whole ORP pipeline for a list of circuits, using the pieces built in 2.1-2.7.

    parity postselection -> detectors -> Delta ranking -> O_k / f_k / sig2_k -> O_ref -> k*

    Returns one `obs_k` dict per circuit, each keyed by (site, rep).
    """
    # TODO: chain the functions you wrote in 2.1-2.7. Note that `R`, the number of syndrome
    # rounds, is the length of that circuit's syndrome bitstring list - compute_obs_ref needs it.

    return obs_k_list

In [ ]:
# These window values are system-specific: they are retuned for the
# 21-block run in PART 3.4, where 42 detectors make the window much tighter.
n_samples = 1000 # bootstrapping
f_min = 0.05     # for reference computation
f_ref = 0.5      # "
z = 1            # for optimal k selection
target_len = 2   # "

obs_k_list = run_orp(encoded_obs, data_bitstrings_list, syn_bitstrings_list, num_logical_blocks,
                     n_samples=n_samples, f_min=f_min, f_ref=f_ref, z=z, target_len=target_len)

<div class="alert alert-block alert-info">

### **Check your implementation below (5 pts).**

The whole pipeline is re-run on a fixed synthetic dataset and compared end to end: $f_k$, $O_k$, $O_{\rm ref}$ and $k^\star$.



<!-- p1-hints -->
<details>
<summary><b>Stuck on 2.8?</b> (click to expand)</summary>

- **$f_k$ looks wrong** - it must come from the **parity-even** ensemble, so run the 2.1 post-selection inside the pipeline.
- **$O_{\rm ref}$ not derivable from your own arrays** - check the $[f_{\min}, f_{\rm ref}]$ window, and that `R` is the number of syndrome rounds actually present for that circuit.
- **$O_{k^\star} \neq O_k[k^\star]$** - it must be *your* $O_k$ evaluated at *your own* $k^\star$.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex2_7
grade_lab_iceberg_ex2_7(run_orp);

In [ ]:
ref_vals = {site: np.array([state.expectation_value(obs).real for state in states])
            for site, obs in unencoded_obs.items()}
rep_marker = {0: 'd', 1: 'o'}
offset = {0: -0.08, 1: 0.08}          # small horizontal shift so the two representatives don't overlap
steps = np.array(n_steps_list)

fig, (ax_raw, ax_orp) = plt.subplots(1, 2, figsize=(10, 4), dpi=300, sharey=True, layout="constrained")
raw_errs, orp_errs = [], []
for (site, rep), obs in encoded_obs.items():
    # raw: every shot, no postselection; 1-sigma error bar for a ±1-valued observable
    raw = np.array([ba.expectation_values(obs) for ba in data_bitarray_list])
    raw_sig = np.sqrt((1 - raw**2) / np.array([ba.num_shots for ba in data_bitarray_list]))
    # ORP: value at the selected cut level k*, with its bootstrap uncertainty
    orp = np.array([ok[(site, rep)]["O_k_star"] for ok in obs_k_list])
    orp_sig = np.array([np.sqrt(ok[(site, rep)]["sig2_k"][ok[(site, rep)]["k_star"]]) for ok in obs_k_list])

    raw_errs.append(np.abs(raw - ref_vals[site]))
    orp_errs.append(np.abs(orp - ref_vals[site]))
    for ax, err, sig in ((ax_raw, raw_errs[-1], raw_sig), (ax_orp, orp_errs[-1], orp_sig)):
        ax.errorbar(steps + offset[rep], err, yerr=sig, color=color_of[site], marker=rep_marker[rep],
                    linestyle='none', capsize=2, markersize=4, elinewidth=0.8)

# mean absolute error over all observables (NaNs from empty ORP cuts are skipped)
for ax, errs in ((ax_raw, raw_errs), (ax_orp, orp_errs)):
    ax.plot(steps, np.nanmean(errs, axis=0), color='black', linewidth=1.5, zorder=0)
    ax.grid(alpha=0.4)
    ax.set_xlabel('n_steps')
ax_raw.set_title('Raw')
ax_orp.set_title('ORP')
ax_raw.set_ylabel(r'$|\langle \bar Z \rangle - \langle Z \rangle_{\mathrm{ref}}|$')

site_handles = [Patch(facecolor=color_of[s], label=str(s)) for s in lattice.nodes()]
method_handles = [Line2D([], [], color='black', marker=rep_marker[0], linestyle='none', label='rep 0'),
                  Line2D([], [], color='black', marker=rep_marker[1], linestyle='none', label='rep 1'),
                  Line2D([], [], color='black', linewidth=1.5, label='mean')]
ax_orp.legend(handles=site_handles + method_handles, ncol=2, loc='upper left');


# **PART 3: Large-Scale Encoded Experiment**


Put the pieces together and run the large-scale encoded experiment on 136 physical qubits.


### 3.1 Establish Lattice
`FIG2A_GRID` is the paper's Fig. 2a logical grid as `{row: {column: (b, l)}}`: 42 sites on 21
blocks. `FIG2A_MISSING` lists the four nearest-neighbor couplings the placement cannot realize.
`LogicalLattice.from_rows(FIG2A_GRID, omit=FIG2A_MISSING)` gives the 66-edge lattice (21
intra-block, 45 inter-block).


In [ ]:
from utils.iceberg_layout import FIG2A_GRID, FIG2A_MISSING

# TODO: the full system of the paper's Fig. 2a: 21 blocks carrying 42 logical qubits.
# `FIG2A_GRID` is that grid as {row: {column: (block, logical)}}; `FIG2A_MISSING` lists the four
# nearest-neighbor couplings this placement cannot realize, so pass them as `omit`.
placement_L = ...
lattice_L = ...
block_qubits_L = ...

# TODO: the nine sites of the true-vacuum bubble, in (block, logical) labels (see Fig. 2a)
bubble_sites_L = ...

block_to_index_L = {b[0]: i for i, b in enumerate(placement_L.blocks.items())}

num_logical_blocks_L = len(placement_L.blocks)

In [ ]:
plot_placement(placement_L, heron=True);

In [ ]:
plot_logical_graph(lattice_L.to_graph(), bubble_sites=bubble_sites_L);

<div class="alert alert-block alert-info">

### **Check your implementation below (2.5 pts).**



<!-- p1-hints -->
<details>
<summary><b>Stuck on 3.1?</b> (click to expand)</summary>

- **Wrong edge count** - pass `omit=FIG2A_MISSING`. Four nearest-neighbour couplings cannot be realized by this placement, so the lattice has 66 edges, not 70.
- Expected: 42 sites, 21 blocks, 66 edges, 9 bubble sites.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex3_1
grade_lab_iceberg_ex3_1(lattice_L, block_qubits_L, bubble_sites_L, placement_L);

### 3.2 Prepare Circuits

A much larger search than 1.4: 21 orientations $o_b$, 21 relabelings $l_b$, and a bridge choice
for every coupling with more than one route. Give the coordinate descent time to converge.

It matters because one Trotter step costs about 24 layers of two-qubit gates and the reference
time needs eight of them. Depth is where most of the final error comes from, so the shallowest
step you can find is the biggest single lever on the median error that 3.5 and 4.3 are graded on.


In [ ]:
# Note: this is the slow cell of PART 3. The CP-SAT search over the 21-block lattice takes
# roughly 80 seconds and prints nothing while it runs - that is expected, do not interrupt it.
# (Measured: ~78 s on 20 cores, ~89 s throttled to 2 workers. It reaches the same
# depth 24 / 409 CNOTs either way, so a smaller machine does not cost you points.)
# TODO: optimize the Trotter step for the full 21-block lattice, exactly as in 1.4.
# This is the single biggest lever on the final error: every Trotter step pays the depth twice
# over, for eight steps.
schedule_L = ...
A_L, step_L, gsched_L = ...

print(f"\none step: depth {gsched_L.depth} ({gsched_L.status}), {gsched_L.n_cx} CNOTs")

placement_L, block_to_index_L = step_L.place, step_L.block_to_index

<div class="alert alert-block alert-info">

### **Check your implementation below (2.5 pts).**

Banded on the two-qubit depth of one Trotter step.



<!-- p1-hints -->
<details>
<summary><b>Stuck on 3.2?</b> (click to expand)</summary>

- **Depth above 24** - let it run to convergence; the optimizer reaches depth 24 with 409 CNOTs here.
- This takes about **80 seconds** and prints nothing while it works. That is normal - do not interrupt it.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex3_2
grade_lab_iceberg_ex3_2(gsched_L);

In [ ]:
reset_ancillas = False

J = 1
gz_melting = 0.5
gz_localized = 1.5
gx = 0.75
delta_t = 0.5


In [ ]:
qcs_L_melting = []
n_steps_L_list = [0, 1, 2, 3, 4, 5, 6, 7, 8]

for n_steps_L in n_steps_L_list: 
    if n_steps_L > 0: 
        circ_elements_L = ['trotter'] * n_steps_L + ['syndrome']
    else: 
        circ_elements_L = [] # just state preparation 
    qc_L = full_circuit(gsched_L, A_L, bubble_sites_L, circ_elements_L, lattice_L, placement_L, block_to_index_L, gz=gz_melting, barriers=False)
    qcs_L_melting.append(qc_L)
    

### 3.3 Submit Jobs

Two things matter when transpiling. The circuits are already laid out on the paper's placement, so
the transpiler must not choose its own layout or permute qubits -- pin virtual qubit $i$ to
physical qubit $i$ and assert nothing moved. And decompose the SWAPs first, so the pass manager
can fuse them with neighboring gates instead of treating them as opaque three-CNOT blocks.

Nine circuits at 32,000 shots is roughly three minutes of QPU time per regime.


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend('ibm_boston')

In [ ]:
from qiskit_ibm_runtime.executor_sampler import Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# TODO: transpile for the backend at optimization_level=2, pinning virtual qubit i to physical
# qubit i so the circuits stay on the paper's placement. Decompose the SWAPs first.
pm = ...

isa_qcs_L_melting = ...

for isa_qc_L in isa_qcs_L_melting: 
    print(isa_qc_L.depth(filter_function=lambda inst: inst.operation.name in ("cz", "rzz")))

In [ ]:
final_layout = isa_qcs_L_melting[0].layout.final_layout
assert final_layout is None or final_layout.to_permutation(isa_qcs_L_melting[0].qubits) == list(range(backend.num_qubits)), \
    "transpiler permuted qubits"

<div class="alert alert-block alert-info">

### **Check your implementation below (2.5 pts).**



<!-- p1-hints -->
<details>
<summary><b>Stuck on 3.3?</b> (click to expand)</summary>

- **Depth above the reference** - transpile at `optimization_level=2` with the identity `initial_layout`, and **decompose the SWAPs first** so the transpiler can fuse them across steps.
- **Layout permuted** - pass `initial_layout=list(range(backend.num_qubits))`.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex3_3
grade_lab_iceberg_ex3_3(isa_qcs_L_melting, n_steps_L_list);

In [ ]:
# If you've already run the job and just want to load it, put the job id here 
melting_job_id = None   # paste a job id here to reload instead of resubmitting

In [ ]:
n_shots = 32000 # as in the paper

if melting_job_id:
    job_melting = service.job(melting_job_id)
else:
    sampler = Sampler(mode=backend)
    sampler.options.default_shots = n_shots
    job_melting = sampler.run(isa_qcs_L_melting)
    melting_job_id = job_melting.job_id()
    print(melting_job_id)
    
    with open("job_ids.txt", "a") as file:
        file.write(f"\nEncoded melting job id ({backend.name}): {melting_job_id}")
    

In [ ]:
result_L_melting = job_melting.result()

### 3.4 Retrieve Results and Postselect

The ORP pipeline from 2.8, now on hardware data.

One parameter changes substantially: the reference window tightens from
$[f_{\min}, f_{\rm ref}] = [0.05, 0.5]$ to $[0.005, 0.05]$. With 42 detectors instead of 4 even a
few cuts discard most of the ensemble, so "deep but not empty" sits an order of magnitude lower in
surviving fraction. `target_len` goes up too, since there are far more levels for a plateau to
appear in.


In [ ]:
data_bitstrings_list_L_melting = []
data_bitarray_list_L_melting = []

syn_bitstrings_list_L_melting = []

for result in result_L_melting:
    syn_bitstrings = []
    for reg_name, bitarray in result.data.items(): 
        if reg_name == 'data': 
            data_bitstrings = bitarray.get_bitstrings()
            data_bitarray = bitarray
        else: 
            syn_bitstrings.append(bitarray.get_bitstrings())
    
    data_bitstrings_list_L_melting.append(data_bitstrings)
    data_bitarray_list_L_melting.append(data_bitarray)
    
    syn_bitstrings_list_L_melting.append(syn_bitstrings)


In [ ]:
data_bitstrings_ps_list_L_melting, syn_bitstrings_ps_list_L_melting = postselect_parity(data_bitstrings_list_L_melting, syn_bitstrings_list_L_melting)

print(f"parity-even shots per circuit: "
      f"{[len(d) for d in data_bitstrings_ps_list_L_melting]}")

In [ ]:
encoded_obs_L, unencoded_obs_L = build_observables(placement_L, lattice_L, A_L)

In [ ]:
# takes 5-6 minutes to run 

n_samples_L = 1000 # bootstrapping
f_min_L = 0.005    # for reference computation
f_ref_L = 0.05     # "
z_L = 1            # for optimal k selection
target_len_L = 5   # "

# TODO: run the ORP pipeline from 2.8 over the melting-regime results. Note the window
# [f_min_L, f_ref_L] is much tighter here than in the toy system: with 21 blocks there are 42
# detectors, so cutting even a few of them discards a large share of the shots.
obs_k_list_L_melting = ...

<div class="alert alert-block alert-info">

### **Check your implementation below (2.5 pts).**



<!-- p1-hints -->
<details>
<summary><b>Stuck on 3.4?</b> (click to expand)</summary>

- **Wrong register width** - expected 84 data bits and 42 syndrome bits for 21 blocks.
- **Too few shots** - the paper uses 32,000. Fewer shots widen the error bars ORP has to work against.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex3_4
grade_lab_iceberg_ex3_4(data_bitstrings_list_L_melting,
                        syn_bitstrings_list_L_melting, obs_k_list_L_melting);

In [ ]:
from utils.iceberg_viz import site_values, plot_bubble_grid

times_L = [n * delta_t for n in n_steps_L_list]

panels_cmp = {
    "parity only": [site_values(obs_k, field="O_k", k=0) for obs_k in obs_k_list_L_melting],
    "ORP":         [site_values(obs_k) for obs_k in obs_k_list_L_melting],
}
plot_bubble_grid(lattice_L, panels_cmp, times_L, bubble_sites=bubble_sites_L,
                 title=f"$g_z = {gz_melting}$, R = 1");

### 3.5 Compare to MPS

The scored comparison:

$$\text{median over sites of } \left| \langle \bar{Z} \rangle_{\rm meas}
- \langle \bar{Z} \rangle_{\rm MPS} \right| \quad \text{at } t = 4$$

Be clear what the MPS reference is. It is an essentially exact simulation of **the same
Trotterized evolution** your circuits implement -- same order, same $\delta t$, bond dimension
high enough to be converged. So the difference you measure is hardware error plus whatever bias
ORP left behind. It is **not** a reference for the exact continuous-time dynamics: both your
circuit and the MPS carry the same Trotter error, and comparing them cannot see it.

That is the right choice here, because the question is whether error detection improves the
accuracy of a *given* circuit -- not how good the Trotter approximation is.

Both field regimes are scored, 5 points each, so the submission cell comes at the end of 3.6 once
the localized results exist too.


In [ ]:
import time
from utils.iceberg_schedule import snake_order

def mps_z_snapshots(step, bubble_sites, snapshot_steps, delta_t, J, gx, gz,
                    max_bond=1500, cutoff=1e-8, verbose=True):
    """<Z> on every site after each number of Trotter steps in `snapshot_steps`, from ONE quimb
    MPS evolution with the same Trotter order as the encoded circuits (step.trotter_order).

    Returns ({n_steps: {site: <Z>}}, {n_steps: (max bond, seconds so far)}).
    """
    # quimb is only needed to REGENERATE the baseline; the shipped data/mps_baseline_L.npz is
    # enough to run the comparison, so the import stays inside the function.
    import quimb as qu
    import quimb.tensor as qtn

    sites = snake_order(step.lattice)                     # 1D order for the MPS, as in the paper
    idx = {s: k for k, s in enumerate(sites)}
    circ = qtn.CircuitMPS(len(sites), max_bond=max_bond, cutoff=cutoff)
    for s in sites:
        if s not in bubble_sites:
            circ.apply_gate("X", idx[s])
    Z, snaps, info, t0 = qu.pauli("Z"), {}, {}, time.time()

    def record(n):
        psi = circ.psi
        snaps[n] = {s: float(np.real(psi.local_expectation_canonical(Z, (idx[s],)))) for s in sites}
        info[n] = (psi.max_bond(), time.time() - t0)
        if verbose:
            print(f"  n_steps = {n:2d} (t = {n * delta_t:4.1f}): max bond {info[n][0]:5d}, {info[n][1]:7.1f} s")

    if 0 in snapshot_steps:
        record(0)
    for n, terms in enumerate(step.trotter_order(max(snapshot_steps)), start=1):
        for term in terms:
            if term[0] == "x":
                circ.apply_gate("RX", -delta_t * gx, idx[term[1]])
            elif term[0] == "z":
                circ.apply_gate("RZ", -delta_t * gz, idx[term[1]])
            else:
                circ.apply_gate("RZZ", -delta_t * J, idx[term[1]], idx[term[2]])
        if n in snapshot_steps:
            record(n)
    return snaps, info


In [ ]:
import numpy as np

def load_mps_baseline(path="data/mps_baseline_L.npz"):
    """{gz: {n_steps: {site: <Z>}}} from the MPS baseline, which ships as a .npz (not a pickle).

    Regenerate it from the original baselines with `python dev/make_mps_baseline_npz.py`.
    """
    with np.load(path, allow_pickle=False) as d:
        sites = [tuple(int(x) for x in row) for row in d["sites"]]
        steps = [int(n) for n in d["n_steps"]]
        return {gz: {n: {s: float(d[key][i][j]) for j, s in enumerate(sites)}
                     for i, n in enumerate(steps)}
                for gz, key in ((0.5, "g0p5"), (1.5, "g1p5"))}

mps_baseline_L = load_mps_baseline()
mps_L_melting = mps_baseline_L

In [ ]:
panels_cmp = {
    "MPS":         [mps_L_melting[gz_melting][n] for n in n_steps_L_list],
    "ORP":         [site_values(ok) for ok in obs_k_list_L_melting],
    "parity only": [site_values(ok, field="O_k", k=0) for ok in obs_k_list_L_melting],
}
plot_bubble_grid(lattice_L, panels_cmp, times_L, bubble_sites=bubble_sites_L, title=f"$g_z = {gz_melting}$")


In [ ]:
from utils.iceberg_viz import site_errors, plot_cross_sections

n = 8
i = n_steps_L_list.index(n)
panels = {
    f"$g_z = {gz_melting}$": {"encoded": site_values(obs_k_list_L_melting[i]),
                    "encoded_err": site_errors(obs_k_list_L_melting[i]),
                    "mps": mps_L_melting[gz_melting][n]},
}
plot_cross_sections(lattice_L, panels, row=3, col=3, title=f"t = {n * delta_t:g}, R = 1, {backend.name}");


### 3.6 Repeat for Localized Regime ($g_z = 1.5$)

The same pipeline on the other side of the phase diagram. At $g_z = 0.5$ the longitudinal field is
weak and the true-vacuum bubble spreads; at $g_z = 1.5$ it is strong enough to hold the bubble in
place and the dynamics stay localized.

Running both is not padding: a postselection scheme that works in one regime and not the other has
probably been tuned to the first. 5 points each.


In [ ]:
gz_localized = 1.5

qcs_L_localized = []

for n_steps_L in n_steps_L_list: 
    if n_steps_L > 0: 
        circ_elements_L = ['trotter'] * n_steps_L + ['syndrome']
    else: 
        circ_elements_L = [] # just state preparation
    qc_L = full_circuit(gsched_L, A_L, bubble_sites_L, circ_elements_L, lattice_L, placement_L, block_to_index_L, gz=gz_localized, barriers=False)
    qcs_L_localized.append(qc_L)
    

In [ ]:
isa_qcs_L_localized = [pm.run(qc_L.decompose(gates_to_decompose=["swap"])) for qc_L in qcs_L_localized]

for isa_qc_L in isa_qcs_L_localized: 
    print(isa_qc_L.depth(filter_function=lambda inst: inst.operation.name in ("cz", "rzz")))

In [ ]:
final_layout = isa_qcs_L_localized[0].layout.final_layout
assert final_layout is None or final_layout.to_permutation(isa_qcs_L_localized[0].qubits) == list(range(backend.num_qubits)), \
    "transpiler permuted qubits"

In [ ]:
# If you've already run the job and just want to load it, put the job id here 
 
localized_job_id = None   # paste a job id here to reload instead of resubmitting

In [ ]:
n_shots = 32000 # as in the paper

if localized_job_id:
    job_localized = service.job(localized_job_id)
else:
    sampler = Sampler(mode=backend)
    sampler.options.default_shots = n_shots
    job_localized = sampler.run(isa_qcs_L_localized)
    localized_job_id = job_localized.job_id()
    print(localized_job_id)
    
    with open("job_ids.txt", "a") as file:
        file.write(f"\nEncoded localized job id ({backend.name}): {localized_job_id}")

In [ ]:
result_L_localized = job_localized.result()

In [ ]:
data_bitstrings_list_L_localized = []
data_bitarray_list_L_localized = []

syn_bitstrings_list_L_localized = []

for result in result_L_localized:
    syn_bitstrings = []
    for reg_name, bitarray in result.data.items(): 
        if reg_name == 'data': 
            data_bitstrings = bitarray.get_bitstrings()
            data_bitarray = bitarray
        else: 
            syn_bitstrings.append(bitarray.get_bitstrings())
    
    data_bitstrings_list_L_localized.append(data_bitstrings)
    data_bitarray_list_L_localized.append(data_bitarray)
    
    syn_bitstrings_list_L_localized.append(syn_bitstrings)


In [ ]:
data_bitstrings_ps_list_L_localized, syn_bitstrings_ps_list_L_localized = postselect_parity(data_bitstrings_list_L_localized, syn_bitstrings_list_L_localized)

In [ ]:
# TODO: the same ORP pipeline, now for the localized regime.
obs_k_list_L_localized = ...

In [ ]:
mps_L_localized = mps_baseline_L

In [ ]:
n = 8
i = n_steps_L_list.index(n)
panels = {
    f"$g_z = {gz_melting}$": {"encoded": site_values(obs_k_list_L_melting[i]),
                          "encoded_err": site_errors(obs_k_list_L_melting[i]),
                          "mps": mps_L_melting[gz_melting][n]},
    f"$g_z = {gz_localized}$": {"encoded": site_values(obs_k_list_L_localized[i]),
                            "encoded_err": site_errors(obs_k_list_L_localized[i]),
                            "mps": mps_L_localized[gz_localized][n]},
}
plot_cross_sections(lattice_L, panels, row=3, col=3, title=f"t = {n * delta_t:g}, R = 1, {backend.name}");


In [ ]:
panels_cmp = {
    "MPS":         [mps_L_melting[gz_melting][n] for n in n_steps_L_list],
    "ORP":         [site_values(ok) for ok in obs_k_list_L_melting],
    "parity only": [site_values(ok, field="O_k", k=0) for ok in obs_k_list_L_melting],
}
plot_bubble_grid(lattice_L, panels_cmp, times_L, bubble_sites=bubble_sites_L, title=f"$g_z = {gz_melting}$");


In [ ]:
panels_cmp = {
    "MPS":         [mps_L_localized[gz_localized][n] for n in n_steps_L_list],
    "ORP":         [site_values(ok) for ok in obs_k_list_L_localized],
    "parity only": [site_values(ok, field="O_k", k=0) for ok in obs_k_list_L_localized],
}
plot_bubble_grid(lattice_L, panels_cmp, times_L, bubble_sites=bubble_sites_L, title=f"$g_z = {gz_localized}$");


In [ ]:
from utils.iceberg_viz import site_values

# NB: these are named enc_ref_* because the error-CDF cell in 4.3 binds
# enc_melting / enc_localized to a per-depth LIST for abs_errors.

i_ref = n_steps_L_list.index(8)          # the reference time t = 4

enc_ref_melting   = site_values(obs_k_list_L_melting[i_ref])
enc_ref_localized = site_values(obs_k_list_L_localized[i_ref])

# the figure of merit: the median over sites of |<Zbar>_meas - <Zbar>_MPS|
median_melting = float(np.median([abs(enc_ref_melting[s] - mps_L_melting[gz_melting][8][s])
                                  for s in enc_ref_melting]))
median_localized = float(np.median([abs(enc_ref_localized[s] - mps_L_localized[gz_localized][8][s])
                                    for s in enc_ref_localized]))
print(f"median |err| vs MPS:  g_z = 0.5: {median_melting:.4f}   g_z = 1.5: {median_localized:.4f}")

<div class="alert alert-block alert-info">

### **Check your implementation below (10 pts total).**

These are **self-reported hardware numbers**: the grader recomputes the figure of merit from the per-site values you submit and compares it against its own MPS reference. Both regimes are scored, 5 pts each.

Each regime is banded against what a bare run reaches there, not on one absolute scale, because the two carry very different signal: at $t = 4$ the median $|\langle \bar{Z} \rangle|$ is about 0.55 for $g_z = 0.5$ and 0.19 for $g_z = 1.5$. Reproducing the reference run earns full marks; reducing the error further is PART 5.1.



<!-- p1-hints -->
<details>
<summary><b>Stuck on 3.5 / 3.6?</b> (click to expand)</summary>

- **Median outside the bands** - check the site ordering of your $\langle \bar{Z} \rangle$ values, and that you are comparing at the same Trotter depth ($t = 4$, i.e. `n_steps = 8`).
- **Median below 0.01** - that is below anything reachable on hardware at 32,000 shots. Submit your **measured** values, not the MPS reference.
- **Claimed median ≠ recomputed** - the grader recomputes the median from your per-site values and tells you both numbers.

</details>

</div>


In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex3_5
grade_lab_iceberg_ex3_5(enc_ref_melting, enc_ref_localized,
                        median_melting, median_localized);

# **PART 4: Unencoded Baseline**


The same Trotterized MFIM simulation directly on $N = 42$ **physical** qubits, *without* the
Iceberg code. This is the baseline that decides whether the encoded run, with all its overhead,
actually helped.


### 4.1 Build and transpile the unencoded circuits


Use `unencoded_trotter_circuit` from PART 1 to build one circuit per entry of `n_steps_L_list`,
for each value of $g_z$. Keep the same gate order, so both arms carry the same Trotter error.


In [ ]:
# TODO: build the unencoded circuits with the function from 1.9, one per Trotter depth in
# `n_steps_L_list`, for both field regimes. Use `step_L.trotter_order(n)` so the unencoded run
# implements the SAME Trotter order as the encoded one - otherwise the comparison also measures
# a difference in Trotter error, not just in hardware error.
qcs_unenc_melting = ...

qcs_unenc_localized = ...

Transpile for hardware. The authors do not state their layout for the unencoded runs, so let the
transpiler choose one -- but use the same layout for every circuit, so the same physical qubits are
used throughout.


In [ ]:
from qiskit.transpiler import generate_preset_pass_manager

# TODO: transpile the unencoded circuits. The authors do not specify a layout for the unencoded
# experiment, so let the transpiler choose one - but choose it ONCE, from the deepest circuit,
# and reuse it for every depth so that all the points come from the same physical qubits.
pm_layout = ...
isa_deepest = ...
layout_unenc = ...

pm_unenc = ...
isa_qcs_unenc_melting = ...
isa_qcs_unenc_localized = ...

Before submitting the jobs, let's compare to the encoded ISA circuits -- specifically, let's look at the two-qubit depth and the number of two-qubit gates.

In [ ]:
TWO_Q = lambda inst: len(inst.qubits) == 2
def two_q_stats(isa):
    return isa.depth(TWO_Q), sum(len(inst.qubits) == 2 for inst in isa.data)

for n, isa_unenc, isa_enc in zip(n_steps_L_list, isa_qcs_unenc_melting, isa_qcs_L_melting):
    d_unenc, n2q_unenc = two_q_stats(isa_unenc)
    d_enc, n2q_enc = two_q_stats(isa_enc)
    print(f"n_steps = {n:2d}: 2q depth {d_unenc:4d} (unencoded) vs. {d_enc:4d} (encoded), 2q gates {n2q_unenc:5d} (unencoded) vs. {n2q_enc:5d} (encoded)")

<div class="alert alert-block alert-info">

### **Check your implementation below (2.5 pts).**



<!-- p1-hints -->
<details>
<summary><b>Stuck on 4.1?</b> (click to expand)</summary>

- **Wrong qubit count** - one circuit per Trotter depth, all sharing one 42-qubit layout.
- Transpile the deepest circuit first, then reuse *its* layout for every other depth, so each time step uses the same physical qubits.

</details>

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex4_1
grade_lab_iceberg_ex4_1(isa_qcs_unenc_melting, n_steps_L_list);

As the Trotter depth grows the unencoded circuits come out *deeper* than the encoded ones. Two
reasons: the encoded gate order was heavily optimized and the code blocks were placed so their
logical couplings are nearly all physical neighbors, while the unencoded circuit needs many SWAPs
to embed a square lattice into heavy-hex.


### 4.2 Submit and retrieve the unencoded jobs


In [ ]:
# If you've already run the jobs and just want to load them, put the job ids here 

unenc_melting_job_id = None   # paste a job id here to reload instead of resubmitting
unenc_localized_job_id = None   # paste a job id here to reload instead of resubmitting

In [ ]:
n_shots_unenc = 32000 # same as the encoded runs, as in the paper

if unenc_melting_job_id and unenc_localized_job_id:
    job_unenc_melting = service.job(unenc_melting_job_id)
    job_unenc_localized = service.job(unenc_localized_job_id)
else:
    sampler = Sampler(mode=backend)
    sampler.options.default_shots = n_shots_unenc
    job_unenc_melting = sampler.run(isa_qcs_unenc_melting)
    unenc_melting_job_id = job_unenc_melting.job_id()
    print(unenc_melting_job_id)
    
    with open("job_ids.txt", "a") as file:
        file.write(f"\nUnencoded melting job id ({backend.name}): {unenc_melting_job_id}")
    
    job_unenc_localized = sampler.run(isa_qcs_unenc_localized)
    unenc_localized_job_id = job_unenc_localized.job_id()
    print(unenc_localized_job_id)
    
    with open("job_ids.txt", "a") as file:
        file.write(f"\nUnencoded localized job id ({backend.name}): {unenc_localized_job_id}")

Once the jobs have made it through the queue, retrieve the results below. 

In [ ]:
result_unenc_melting = job_unenc_melting.result()
result_unenc_localized = job_unenc_localized.result()

In [ ]:
unenc_L_melting = [{site: float(result.data.meas.expectation_values(obs)) for site, obs in unencoded_obs_L.items()} 
                   for result in result_unenc_melting]

unenc_L_localized = [{site: float(result.data.meas.expectation_values(obs)) for site, obs in unencoded_obs_L.items()} 
                     for result in result_unenc_localized]

### 4.3 Compare encoded against unencoded


In [ ]:
from utils.iceberg_viz import unencoded_errors

n = 8
i = n_steps_L_list.index(n)
panels = {
    f"$g_z = {gz_melting}$": {"encoded": site_values(obs_k_list_L_melting[i]),
                          "encoded_err": site_errors(obs_k_list_L_melting[i]),
                          "unencoded": unenc_L_melting[i],
                          "unencoded_err": unencoded_errors(unenc_L_melting[i], n_shots_unenc),
                          "mps": mps_L_melting[gz_melting][n]},
    f"$g_z = {gz_localized}$": {"encoded": site_values(obs_k_list_L_localized[i]),
                            "encoded_err": site_errors(obs_k_list_L_localized[i]),
                            "unencoded": unenc_L_localized[i],
                            "unencoded_err": unencoded_errors(unenc_L_localized[i], n_shots_unenc),
                            "mps": mps_L_localized[gz_localized][n]},
}
plot_cross_sections(lattice_L, panels, row=3, col=3, title=f"t = {n * delta_t:g}, R = 1, {backend.name}");


In [ ]:
from utils.iceberg_viz import abs_errors, plot_error_cdfs

n_used = [n for n in n_steps_L_list if n > 0]     # leave out t = 0 (no time evolution yet)

enc_melting = [site_values(obs_k) for obs_k in obs_k_list_L_melting]       # ORP, reps averaged
enc_localized = [site_values(obs_k) for obs_k in obs_k_list_L_localized]

panels = {
    f"$g_z = {gz_melting}$": {
        "encoded": abs_errors(enc_melting, mps_L_melting[gz_melting], n_steps_L_list, n_used),
        "unencoded": abs_errors(unenc_L_melting, mps_L_melting[gz_melting], n_steps_L_list, n_used)},
    f"$g_z = {gz_localized}$": {
        "encoded": abs_errors(enc_localized, mps_L_localized[gz_localized], n_steps_L_list, n_used),
        "unencoded": abs_errors(unenc_L_localized, mps_L_localized[gz_localized], n_steps_L_list, n_used)},
}
plot_error_cdfs(panels, title=f"{backend.name}, R = 1, $t \\leq {max(n_used) * delta_t:g}$");


<div class="alert alert-block alert-info">

### **Check your implementation below (7.5 pts).**

These are **self-reported hardware numbers**. The grader cannot re-run your job, so it recomputes the figure of merit from the per-site values you submit and compares that against its own MPS reference.

The encoded-versus-unencoded comparison is graded in the **melting** regime (5 pts, scaled by how far the encoded median beats the unencoded one). The localized regime is graded on being a sound pair of measurements (2.5 pts), not on which arm wins -- see 4.4 for why.



<!-- p1-hints -->
<details>
<summary><b>Stuck on 4.3?</b> (click to expand)</summary>

- **Encoded loses in the melting regime** - check the ORP cut, and that both arms ran the same Trotter order and the same depth.
- In the **localized** regime the encoded arm is *not* required to win: full marks there need a sound pair of measurements, not a victory.

</details>

</div>


In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex4_2
grade_lab_iceberg_ex4_2(enc_ref_melting, unenc_L_melting[i_ref],
                        enc_ref_localized, unenc_L_localized[i_ref]);

### 4.4 What the comparison tells us

<div class="alert alert-block alert-info">

**The code helped where the measurement can tell.** In the melting regime the encoded median
comes in below the unencoded one, which is the result the challenge exists to test: detecting
errors and spending shots on postselection buys more accuracy than it costs in statistics --
even though the encoded circuit runs on 136 qubits instead of 42, and even though its logical
operations are *not* fault tolerant.

**The localized regime cannot settle it.** The figure of merit is an absolute error, and at
$t = 4$ the localized regime carries a median $|\langle \bar{Z} \rangle|$ of only 0.19 against
the melting regime's 0.55. The encoded arm arrives there having spent all but a per cent or so
of its shots on ORP, while the unencoded arm keeps all 32,000. At that signal level the
difference between the two arms is dominated by the shot penalty rather than by bias, and it can
rank the noisier arm first. Do not read an unencoded win there as the code failing -- read it as
the metric running out of signal. Say which way yours came out and why.

**The baseline was not disadvantaged.** The unencoded circuits are *deeper* in two-qubit gates at
the same Trotter depth, because a square lattice of 42 sites does not embed in heavy-hex without
SWAPs, while the code blocks were placed so their logical couplings are nearly all physical
neighbors. On a square-lattice device the baseline would be much shallower -- which is what PART
5.2 measures.

</div>


# **PART 5: Improvements**

PARTS 3 and 4 left two loose ends, and this part picks up both.

**Nothing was done about the noise.** Everything so far ran on a bare sampler, with no error
suppression of any kind. 5.1 is where you add some; the choice of technique is yours.

**The baseline paid a routing cost the physics does not require.** As PART 4.4 noted, the
unencoded circuits came out *deeper* than the encoded ones, because a 42-site square lattice
does not embed in heavy-hex without SWAPs. A fair verdict on "does error detection help" needs a
baseline that does not pay that toll.

### How PART 5 is graded (10 pts)

| | | Points |
|---|---|---:|
| 5.1 | Reduce the error on both arms, by whatever means you choose | 5 |
| 5.2 | The unencoded baseline re-run on a square-lattice device | 5 |

The figure of merit is unchanged: the median over sites of
$|\langle \bar{Z} \rangle_{\rm meas} - \langle \bar{Z} \rangle_{\rm MPS}|$ at $t = 4$. 5.1 is
scored mostly on the *improvement* over your PART 3.5 median, so keep that number on hand.

Both graded exercises only need the **deepest point** ($n_{\rm steps} = 8$), so the extra QPU
cost is less than a minute.

### 5.1 Reduce the error on both arms

Anything that lowers the median error counts here. The grader does not inspect which approach
you took -- it scores the improvement over your PART 3.5 number. Apply the same treatment to
both arms so the comparison stays fair.

Two things about these circuits are worth knowing when you choose: most of the 136 qubits sit
idle while one block runs its syndrome gadget, and every syndrome bit is a measurement, so
readout bias feeds straight into the detectors. Approaches that cost extra circuits or extra
shots will run into the challenge's QPU budget.


In [ ]:
from qiskit_ibm_runtime.executor_sampler import Sampler

# TODO: pick your approach and configure the sampler for it. Record what you applied below --
# free-form, unscored, but the write-up needs it.
mitigation_applied = ...

sampler_mit = Sampler(mode=backend)
sampler_mit.options.default_shots = n_shots
# TODO: configure sampler_mit for whatever you chose


In [ ]:
def unpack_encoded(result):
    """(data_bitstrings_list, syn_bitstrings_list) from a Sampler result of encoded circuits."""
    data_list, syn_list = [], []
    for pub in result:
        syn = []
        for reg_name, bitarray in pub.data.items():
            if reg_name == "data":
                data_list.append(bitarray.get_bitstrings())
            else:
                syn.append(bitarray.get_bitstrings())
        syn_list.append(syn)
    return data_list, syn_list

In [ ]:
# TODO: submit the deepest encoded circuit and the deepest unencoded circuit through
# `sampler_mit`, following the same load-or-submit pattern as PART 3.3 so you do not resubmit
# every time the notebook is re-run. Save the job ids to job_ids.txt.
# mit_enc_job_id = ""
# mit_unenc_job_id = ""

job_mit_enc = ...
job_mit_unenc = ...

In [ ]:
# TODO: pull both results apart, run the encoded one through `run_orp` (same parameters as
# PART 3.4), and build {site: <Zbar>} for each arm at t = 4. Then compute each median against
# the MPS baseline, and compare the encoded one with your unmitigated `median_melting`.
enc_mitigated = ...
unenc_mitigated = ...

median_enc_mit = ...
median_unenc_mit = ...

<div class="alert alert-block alert-info">

### **Check your implementation below (5 pts).**

3.0 pts for improving on your PART 3.5 median, 1.5 for how accurate the improved encoded result is, 0.5 for a complete unencoded arm. What you applied is recorded but not scored. These are self-reported numbers.



<!-- p1-hints -->
<details>
<summary><b>Stuck on 5.1?</b> (click to expand)</summary>

- **No improvement recorded** - set `options.dynamical_decoupling.enable` and/or `options.twirling.enable_measure` on the sampler before submitting.
- The technique is yours to choose and is **not** scored - the improvement over your PART 3.5 median is.

</details>

</div>


In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex5_1
grade_lab_iceberg_ex5_1(mitigation_applied, enc_mitigated, unenc_mitigated,
                        median_enc_unmitigated=median_melting);


### 5.2 The unencoded baseline on a square lattice

The model is a square lattice of 42 sites. Heavy-hex has degree 3 and no square faces, so the
transpiler routes the missing couplings with SWAPs -- which is why the unencoded circuits ended
up at two-qubit depth 319 while the encoded ones, whose blocks were *placed* so their logical
couplings are physical neighbors, stayed at 168.

On a device whose coupling map already contains squares the lattice embeds directly and each
Trotter step needs only a few `rzz` layers with no routing at all. That makes the baseline both
shallower and more accurate -- a genuinely harder opponent for the encoded run.

Transpile first and look at the numbers before spending any QPU time: the depth comparison is
free and deterministic, and it carries most of the marks.


In [ ]:
TWO_Q = lambda inst: len(inst.qubits) == 2

# TODO: pick a Nighthawk (square-lattice) backend - ibm_miami or ibm_phoenix - and transpile the
# unencoded circuits for it. Then measure the two-qubit depth and two-qubit gate count of the
# deepest circuit on both devices, so the comparison is like for like.
backend_square = ...
isa_qcs_square_melting = ...

depth_heavyhex = ...
depth_square = ...
gates_heavyhex = ...
gates_square = ...

In [ ]:
# TODO: run the deepest unencoded circuit on the square-lattice device and build
# {site: <Zbar>} from the result, then take its median error against the MPS baseline.
# 1.5 of the 5 points need this run - the rest you already earned by transpiling.
# sq_job_id = ""

unenc_square = ...
median_square = ...

<div class="alert alert-block alert-info">

### **Check your implementation below (5 pts).**

3.5 pts come from the transpiled depth and gate counts, which are deterministic; 1.5 pts need the square-lattice run itself.



<!-- p1-hints -->
<details>
<summary><b>Stuck on 5.2?</b> (click to expand)</summary>

- **No depth reduction** - on a square lattice each Trotter step needs only a few `rzz` layers with no routing. Did you transpile against the **Nighthawk** coupling map?
- Measure depth and gate counts for the *same* circuit on both devices, so the comparison is like for like.

</details>

</div>


In [ ]:
from qc_grader.challenges.qdc_2026 import grade_lab_iceberg_ex5_2
grade_lab_iceberg_ex5_2(backend_square.name, depth_heavyhex, depth_square,
                        gates_heavyhex, gates_square, unenc_square);

<div class="alert alert-block alert-success">

## **Congratulations on making it this far!** 🎉

You have built a partially fault-tolerant simulation end to end: 42 logical qubits in $[[4,2,2]]$
blocks on 136 physical qubits, a scheduled Trotter step, fault-tolerant syndrome extraction,
observable-ranked postselection, and an unencoded baseline run under the same conditions to check
whether any of it helped.

</div>


<div class="alert alert-block alert-warning">

## **Stretch Goals**

The exercises stop at one reference time and one syndrome round. The paper, and the physics, go
further:

- **More syndrome rounds.** $R = 1$ is the cheapest possible schedule. With $R = 2$ the detectors
  multiply and ORP has more to rank -- at the cost of two more rounds of gadget depth.
- **Longer evolution.** $t = 4$ is where the MPS reference was computed; the bubble has not
  finished decaying. Pushing further needs a bigger `max_bond` for the reference.
- **Reset the ancillas.** Everything here ran with `reset_ancillas = False`, as the paper does.
  Resetting restores $|0\rangle$ and removes the sign flip a syndrome of 1 imposes on rotations
  routed through that ancilla -- at the cost of a reset per block per round.
- **A different code.** The $[[k+2, k, 2]]$ family generalizes Iceberg. Larger $k$ amortizes the
  two ancillas over more logical qubits, so the overhead falls -- but a block then spans more of
  the device, and the placement problem gets harder.
- **Further error mitigation** on top of 5.1, if you have QPU time to spare.
- **Alternative mappings** of the blocks to heavy-hex, or to the square lattice of 5.2.

If a change improves your results, **overwrite your best numbers and re-submit** -- the grader
keeps the highest score per exercise, so a later worse attempt costs you nothing.

</div>

In [ ]:
# Re-submit your best results. The highest score per exercise is the one that counts, so a
# later, worse attempt never costs you anything.
#
# These default to the run you already have above. If a stretch goal gave you better numbers,
# point them at that run instead and re-run this cell.
enc_best_melting,    enc_best_localized    = enc_ref_melting, enc_ref_localized
median_best_melting, median_best_localized = median_melting, median_localized
unenc_best_melting,  unenc_best_localized  = unenc_L_melting[i_ref], unenc_L_localized[i_ref]

grade_lab_iceberg_ex3_5(enc_best_melting, enc_best_localized, median_best_melting, median_best_localized);
grade_lab_iceberg_ex4_2(enc_best_melting, unenc_best_melting, enc_best_localized, unenc_best_localized);